# Notebook 27 - DINOv2-S Student Baseline

## Objective

This notebook begins the transition from the earlier ResNet18 student to a stronger DINOv2-S-based writer-verification model.

Notebook 26 established a large clean representation advantage for DINOv2-S under the internal 145-writer fit / 36-writer selection protocol. The adapted DINOv2-S representation reached a cross-script macro AUC of approximately 0.8431, compared with approximately 0.7828 for the strongest compact ResNet18 WGC model.

However, that result used fit-only PCA followed by supervised writer-LDA. It therefore does not yet represent a practical trainable neural writer-verification student.

## Research Question

> Can a lightweight trainable projection on top of frozen DINOv2-S features learn an effective cross-script writer embedding without using LDA as the final verifier?

## Baseline Architecture

The first DINOv2-S student is intentionally simple:

**frozen DINOv2-S 384-D feature → trainable linear 144-D projection → L2 normalization → cosine verification**

The DINOv2-S backbone remains frozen in this baseline.

The 144-D projection dimension is fixed structurally from the 145 fit writers, matching the maximum writer-discriminant dimensionality observed in the clean fit-only LDA audit. It is not selected using selection outcomes.

## Training Protocol

The clean internal protocol is preserved:

- Fit: 145 writers
- Selection: 36 writers
- Monitor: untouched
- Historical validation: unused
- Official test: unused

The existing writer-centric fit schedule will be reused.

The initial student baseline will use the matched writer-verification objective without WGC, teacher distillation, script adversarial training, or other auxiliary losses.

This isolates the effect of replacing the ResNet18 representation with DINOv2-S.

## Decision Strategy

The first goal is not to beat the PCA/LDA representation immediately.

The experiment will determine:

1. whether a trainable neural projection can exploit frozen DINOv2-S writer information;
2. how much of the PCA/LDA upper reference it recovers;
3. whether further DINOv2-S adaptation is justified.

Only after establishing this baseline will partial transformer fine-tuning or a new cross-script-specific method be considered.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, roc_curve

In [2]:
ROOT = Path(
    "/home/arijit/Documents/handwriting-cross-script-research"
)

SPLIT_PATH = (
    ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

FIT_SCHEDULE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "fit_writer_batch_schedule_seed42.csv"
)

DINO_S_PATH = (
    ROOT
    / "reports"
    / "dinov2_efficiency_frontier"
    / "dinov2_vits14_reg_embeddings.npz"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
)

CHECKPOINT_DIR = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_df = pd.read_csv(
    SPLIT_PATH
)

role_df = pd.read_csv(
    ROLE_PATH
)

fit_schedule_df = pd.read_csv(
    FIT_SCHEDULE_PATH
)

role_column_candidates = []

for column in role_df.columns:
    values = set(
        role_df[
            column
        ]
        .astype(str)
        .str.lower()
        .unique()
    )

    if {
        "fit",
        "selection",
    }.issubset(
        values
    ):
        role_column_candidates.append(
            column
        )

if len(
    role_column_candidates
) != 1:
    raise RuntimeError(
        "Could not uniquely identify the fit/selection role column."
    )

ROLE_COLUMN = (
    role_column_candidates[
        0
    ]
)

dino_s_archive = np.load(
    DINO_S_PATH,
    allow_pickle=False,
)

required_dino_keys = {
    "development_embeddings",
    "development_filenames",
    "development_writers",
}

if not required_dino_keys.issubset(
    dino_s_archive.files
):
    raise RuntimeError(
        "Required DINOv2-S development arrays are missing."
    )

development_embedding_shape = list(
    dino_s_archive[
        "development_embeddings"
    ].shape
)

development_filename_shape = list(
    dino_s_archive[
        "development_filenames"
    ].shape
)

development_writer_shape = list(
    dino_s_archive[
        "development_writers"
    ].shape
)

role_counts = {
    str(
        key
    ): int(
        value
    )
    for key, value in (
        role_df[
            ROLE_COLUMN
        ]
        .value_counts()
        .to_dict()
        .items()
    )
}

schedule_epoch_count = int(
    fit_schedule_df[
        "epoch"
    ].nunique()
)

schedule_batch_count = int(
    fit_schedule_df[
        [
            "epoch",
            "batch",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
)

schedule_writer_count = int(
    fit_schedule_df[
        "writer"
    ].nunique()
)

schedule_rows_per_epoch = (
    fit_schedule_df
    .groupby(
        "epoch"
    )
    .size()
)

schedule_batches_per_epoch = (
    fit_schedule_df[
        [
            "epoch",
            "batch",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "epoch"
    )
    .size()
)

schedule_unique_writers_per_epoch = (
    fit_schedule_df
    .groupby(
        "epoch"
    )[
        "writer"
    ]
    .nunique()
)

dinov2s_student_resource_audit = {
    "split_path_exists": bool(
        SPLIT_PATH.exists()
    ),
    "role_path_exists": bool(
        ROLE_PATH.exists()
    ),
    "fit_schedule_path_exists": bool(
        FIT_SCHEDULE_PATH.exists()
    ),
    "dinov2s_path_exists": bool(
        DINO_S_PATH.exists()
    ),
    "split_rows": int(
        len(
            split_df
        )
    ),
    "role_rows": int(
        len(
            role_df
        )
    ),
    "role_column": (
        ROLE_COLUMN
    ),
    "role_counts": (
        role_counts
    ),
    "dinov2s_artifact": str(
        DINO_S_PATH.relative_to(
            ROOT
        )
    ),
    "dinov2s_archive_keys": list(
        dino_s_archive.files
    ),
    "development_embedding_shape": (
        development_embedding_shape
    ),
    "development_filename_shape": (
        development_filename_shape
    ),
    "development_writer_shape": (
        development_writer_shape
    ),
    "raw_feature_dimension": int(
        development_embedding_shape[
            1
        ]
    ),
    "schedule_rows": int(
        len(
            fit_schedule_df
        )
    ),
    "schedule_epochs": int(
        schedule_epoch_count
    ),
    "schedule_batches": int(
        schedule_batch_count
    ),
    "schedule_writers": int(
        schedule_writer_count
    ),
    "schedule_columns": list(
        fit_schedule_df.columns
    ),
    "all_epochs_have_145_writer_rows": bool(
        (
            schedule_rows_per_epoch
            == 145
        ).all()
    ),
    "all_epochs_have_18_batches": bool(
        (
            schedule_batches_per_epoch
            == 18
        ).all()
    ),
    "all_epochs_have_145_unique_writers": bool(
        (
            schedule_unique_writers_per_epoch
            == 145
        ).all()
    ),
    "raw_embeddings_reused": True,
    "backbone_forward_performed": False,
    "projection_created": False,
    "training_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_student_resource_audit.json",
    "w",
) as file:
    json.dump(
        dinov2s_student_resource_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_student_resource_audit,
        indent=2,
    )
)

print(
    "\nSchedule batch sizes:"
)

print(
    fit_schedule_df[
        [
            "epoch",
            "batch",
            "batch_size",
        ]
    ]
    .drop_duplicates()[
        "batch_size"
    ]
    .value_counts()
    .sort_index()
    .to_string()
)

if (
    len(
        split_df
    ) != 1900
    or len(
        role_df
    ) != 724
    or role_counts.get(
        "fit",
        0
    ) != 580
    or role_counts.get(
        "selection",
        0
    ) != 144
    or development_embedding_shape
    != [
        904,
        384,
    ]
    or schedule_epoch_count
    != 10
    or schedule_batch_count
    != 180
    or schedule_writer_count
    != 145
    or not (
        schedule_rows_per_epoch
        == 145
    ).all()
    or not (
        schedule_batches_per_epoch
        == 18
    ).all()
    or not (
        schedule_unique_writers_per_epoch
        == 145
    ).all()
):
    raise RuntimeError(
        "DINOv2-S student resource audit failed."
    )

{
  "split_path_exists": true,
  "role_path_exists": true,
  "fit_schedule_path_exists": true,
  "dinov2s_path_exists": true,
  "split_rows": 1900,
  "role_rows": 724,
  "role_column": "internal_role",
  "role_counts": {
    "fit": 580,
    "selection": 144
  },
  "dinov2s_artifact": "reports/dinov2_efficiency_frontier/dinov2_vits14_reg_embeddings.npz",
  "dinov2s_archive_keys": [
    "development_embeddings",
    "development_filenames",
    "development_writers",
    "validation_embeddings",
    "validation_filenames"
  ],
  "development_embedding_shape": [
    904,
    384
  ],
  "development_filename_shape": [
    904
  ],
  "development_writer_shape": [
    904
  ],
  "raw_feature_dimension": 384,
  "schedule_rows": 1450,
  "schedule_epochs": 10,
  "schedule_batches": 180,
  "schedule_writers": 145,
  "schedule_columns": [
    "epoch",
    "batch",
    "slot",
    "writer",
    "batch_size"
  ],
  "all_epochs_have_145_writer_rows": true,
  "all_epochs_have_18_batches": true,
  "al

In [3]:
dino_s_development_embeddings = (
    dino_s_archive[
        "development_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

dino_s_development_filenames = (
    dino_s_archive[
        "development_filenames"
    ]
    .astype(str)
)

dino_s_development_writers = (
    dino_s_archive[
        "development_writers"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

if (
    dino_s_development_embeddings.shape
    != (
        904,
        384,
    )
):
    raise RuntimeError(
        "Unexpected DINOv2-S development embedding shape."
    )

if (
    len(
        np.unique(
            dino_s_development_filenames
        )
    )
    != 904
):
    raise RuntimeError(
        "Duplicate development filenames found in DINOv2-S archive."
    )

dino_s_filename_to_index = {
    filename: int(
        index
    )
    for index, filename in enumerate(
        dino_s_development_filenames
    )
}

fit_rows = (
    role_df[
        role_df[
            ROLE_COLUMN
        ] == "fit"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
    .copy()
)

selection_rows = (
    role_df[
        role_df[
            ROLE_COLUMN
        ] == "selection"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
    .copy()
)

fit_missing_filenames = [
    filename
    for filename in (
        fit_rows[
            "filename"
        ].astype(str)
    )
    if filename
    not in dino_s_filename_to_index
]

selection_missing_filenames = [
    filename
    for filename in (
        selection_rows[
            "filename"
        ].astype(str)
    )
    if filename
    not in dino_s_filename_to_index
]

if (
    len(
        fit_missing_filenames
    ) > 0
    or len(
        selection_missing_filenames
    ) > 0
):
    raise RuntimeError(
        "Fit or selection filenames are missing from DINOv2-S archive."
    )

fit_indices = np.array(
    [
        dino_s_filename_to_index[
            filename
        ]
        for filename in (
            fit_rows[
                "filename"
            ].astype(str)
        )
    ],
    dtype=np.int64,
)

selection_indices = np.array(
    [
        dino_s_filename_to_index[
            filename
        ]
        for filename in (
            selection_rows[
                "filename"
            ].astype(str)
        )
    ],
    dtype=np.int64,
)

fit_raw_features = (
    dino_s_development_embeddings[
        fit_indices
    ]
)

selection_raw_features = (
    dino_s_development_embeddings[
        selection_indices
    ]
)

fit_archive_writers = (
    dino_s_development_writers[
        fit_indices
    ]
)

selection_archive_writers = (
    dino_s_development_writers[
        selection_indices
    ]
)

fit_writer_mismatches = int(
    (
        fit_archive_writers
        != fit_rows[
            "writer"
        ].to_numpy(
            dtype=np.int64
        )
    ).sum()
)

selection_writer_mismatches = int(
    (
        selection_archive_writers
        != selection_rows[
            "writer"
        ].to_numpy(
            dtype=np.int64
        )
    ).sum()
)

fit_writer_set = set(
    fit_rows[
        "writer"
    ].astype(int)
)

selection_writer_set = set(
    selection_rows[
        "writer"
    ].astype(int)
)

schedule_writer_set = set(
    fit_schedule_df[
        "writer"
    ].astype(int)
)

fit_selection_overlap = (
    fit_writer_set
    & selection_writer_set
)

schedule_missing_fit_writers = (
    fit_writer_set
    - schedule_writer_set
)

schedule_extra_writers = (
    schedule_writer_set
    - fit_writer_set
)

fit_writer_feature_map = {}

fit_writer_filename_map = {}

for writer, group in (
    fit_rows
    .groupby(
        "writer",
        sort=True,
    )
):
    group = (
        group
        .sort_values(
            "page_id"
        )
    )

    page_ids = (
        group[
            "page_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    if not np.array_equal(
        page_ids,
        np.array(
            [
                1,
                2,
                3,
                4,
            ],
            dtype=np.int64,
        ),
    ):
        raise RuntimeError(
            f"Invalid page structure for fit writer {writer}."
        )

    row_indices = (
        group
        .index
        .to_numpy(
            dtype=np.int64
        )
    )

    fit_writer_feature_map[
        int(
            writer
        )
    ] = (
        fit_raw_features[
            row_indices
        ]
        .astype(
            np.float32,
            copy=False,
        )
    )

    fit_writer_filename_map[
        int(
            writer
        )
    ] = (
        group[
            "filename"
        ]
        .astype(str)
        .tolist()
    )

selection_filename_feature_map = {
    filename: feature
    for filename, feature in zip(
        selection_rows[
            "filename"
        ].astype(str),
        selection_raw_features,
    )
}

first_batch_schedule = (
    fit_schedule_df[
        (
            fit_schedule_df[
                "epoch"
            ] == 1
        )
        & (
            fit_schedule_df[
                "batch"
            ] == 1
        )
    ]
    .sort_values(
        "slot"
    )
)

first_batch_writers = (
    first_batch_schedule[
        "writer"
    ]
    .astype(int)
    .tolist()
)

first_batch_raw_features = np.stack(
    [
        fit_writer_feature_map[
            writer
        ]
        for writer in (
            first_batch_writers
        )
    ],
    axis=0,
)

fit_raw_norms = np.linalg.norm(
    fit_raw_features,
    axis=1,
)

selection_raw_norms = np.linalg.norm(
    selection_raw_features,
    axis=1,
)

dinov2s_student_alignment_audit = {
    "fit_pages": int(
        len(
            fit_rows
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_set
        )
    ),
    "fit_raw_shape": list(
        fit_raw_features.shape
    ),
    "selection_pages": int(
        len(
            selection_rows
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_set
        )
    ),
    "selection_raw_shape": list(
        selection_raw_features.shape
    ),
    "fit_missing_filenames": int(
        len(
            fit_missing_filenames
        )
    ),
    "selection_missing_filenames": int(
        len(
            selection_missing_filenames
        )
    ),
    "fit_writer_mismatches": int(
        fit_writer_mismatches
    ),
    "selection_writer_mismatches": int(
        selection_writer_mismatches
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_selection_overlap
        )
    ),
    "schedule_missing_fit_writers": int(
        len(
            schedule_missing_fit_writers
        )
    ),
    "schedule_extra_writers": int(
        len(
            schedule_extra_writers
        )
    ),
    "fit_writer_lookup_entries": int(
        len(
            fit_writer_feature_map
        )
    ),
    "selection_filename_lookup_entries": int(
        len(
            selection_filename_feature_map
        )
    ),
    "first_batch_writers": (
        first_batch_writers
    ),
    "first_batch_raw_shape": list(
        first_batch_raw_features.shape
    ),
    "fit_raw_norm_min": float(
        fit_raw_norms.min()
    ),
    "fit_raw_norm_mean": float(
        fit_raw_norms.mean()
    ),
    "fit_raw_norm_max": float(
        fit_raw_norms.max()
    ),
    "selection_raw_norm_min": float(
        selection_raw_norms.min()
    ),
    "selection_raw_norm_mean": float(
        selection_raw_norms.mean()
    ),
    "selection_raw_norm_max": float(
        selection_raw_norms.max()
    ),
    "raw_feature_l2_normalization_performed": False,
    "projection_created": False,
    "training_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_student_raw_alignment_audit.json",
    "w",
) as file:
    json.dump(
        dinov2s_student_alignment_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_student_alignment_audit,
        indent=2,
    )
)

if (
    fit_raw_features.shape
    != (
        580,
        384,
    )
    or selection_raw_features.shape
    != (
        144,
        384,
    )
    or len(
        fit_writer_feature_map
    ) != 145
    or len(
        selection_filename_feature_map
    ) != 144
    or first_batch_raw_features.shape
    != (
        8,
        4,
        384,
    )
    or fit_writer_mismatches
    != 0
    or selection_writer_mismatches
    != 0
    or len(
        fit_selection_overlap
    ) != 0
    or len(
        schedule_missing_fit_writers
    ) != 0
    or len(
        schedule_extra_writers
    ) != 0
    or not np.isfinite(
        fit_raw_features
    ).all()
    or not np.isfinite(
        selection_raw_features
    ).all()
):
    raise RuntimeError(
        "DINOv2-S student raw alignment failed."
    )

{
  "fit_pages": 580,
  "fit_writers": 145,
  "fit_raw_shape": [
    580,
    384
  ],
  "selection_pages": 144,
  "selection_writers": 36,
  "selection_raw_shape": [
    144,
    384
  ],
  "fit_missing_filenames": 0,
  "selection_missing_filenames": 0,
  "fit_writer_mismatches": 0,
  "selection_writer_mismatches": 0,
  "fit_selection_writer_overlap": 0,
  "schedule_missing_fit_writers": 0,
  "schedule_extra_writers": 0,
  "fit_writer_lookup_entries": 145,
  "selection_filename_lookup_entries": 144,
  "first_batch_writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "first_batch_raw_shape": [
    8,
    4,
    384
  ],
  "fit_raw_norm_min": 0.9999998807907104,
  "fit_raw_norm_mean": 1.0,
  "fit_raw_norm_max": 1.0000001192092896,
  "selection_raw_norm_min": 0.9999998807907104,
  "selection_raw_norm_mean": 1.0,
  "selection_raw_norm_max": 1.0000001192092896,
  "raw_feature_l2_normalization_performed": false,
  "projection_created": false,
  "training_p

In [4]:
NOTEBOOK20_PATH = (
    ROOT
    / "notebooks"
    / "20_dinov2_efficiency_frontier.ipynb"
)

with open(
    NOTEBOOK20_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook20 = json.load(
        file
    )

search_terms = [
    "dinov2_vits14_reg",
    "vits14_reg",
    "normalize",
    "norm(",
    "linalg.norm",
    "development_embeddings",
    "savez",
    "forward_features",
    "x_norm_clstoken",
]

matched_cells = []

for cell_index, cell in enumerate(
    notebook20[
        "cells"
    ]
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    matched_terms = [
        term
        for term in (
            search_terms
        )
        if term.lower()
        in source.lower()
    ]

    if len(
        matched_terms
    ) > 0:
        matched_cells.append(
            {
                "cell_index": int(
                    cell_index
                ),
                "matched_terms": (
                    matched_terms
                ),
                "source": (
                    source
                ),
            }
        )

explicit_l2_patterns = [
    "F.normalize",
    "torch.nn.functional.normalize",
    "/ torch.linalg.vector_norm",
    "/ torch.norm",
    "/ np.linalg.norm",
    "embeddings /",
    "embedding /",
]

explicit_l2_matches = []

for item in matched_cells:
    source = (
        item[
            "source"
        ]
    )

    found_patterns = [
        pattern
        for pattern in (
            explicit_l2_patterns
        )
        if pattern.lower()
        in source.lower()
    ]

    if len(
        found_patterns
    ) > 0:
        explicit_l2_matches.append(
            {
                "cell_index": int(
                    item[
                        "cell_index"
                    ]
                ),
                "patterns": (
                    found_patterns
                ),
            }
        )

archive_norms = np.linalg.norm(
    dino_s_development_embeddings,
    axis=1,
)

all_archive_features_unit_norm = bool(
    np.allclose(
        archive_norms,
        1.0,
        atol=1e-5,
    )
)

dinov2s_feature_provenance_audit = {
    "source_notebook": str(
        NOTEBOOK20_PATH.relative_to(
            ROOT
        )
    ),
    "matched_code_cells": int(
        len(
            matched_cells
        )
    ),
    "explicit_l2_pattern_matches": (
        explicit_l2_matches
    ),
    "archive_norm_min": float(
        archive_norms.min()
    ),
    "archive_norm_mean": float(
        archive_norms.mean()
    ),
    "archive_norm_max": float(
        archive_norms.max()
    ),
    "all_archive_features_unit_norm": bool(
        all_archive_features_unit_norm
    ),
    "source_normalization_status": (
        "explicit_normalization_found"
        if len(
            explicit_l2_matches
        ) > 0
        else (
            "unit_norm_archive_but_explicit_source_not_yet_identified"
            if all_archive_features_unit_norm
            else "no_unit_norm_evidence"
        )
    ),
    "training_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_feature_provenance_audit.json",
    "w",
) as file:
    json.dump(
        dinov2s_feature_provenance_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_feature_provenance_audit,
        indent=2,
    )
)

print(
    "\nRelevant Notebook 20 code cells:"
)

for item in matched_cells:
    print(
        "\n"
        + "=" * 80
    )

    print(
        f"Cell index: "
        f"{item['cell_index']}"
    )

    print(
        "Matched terms:",
        item[
            "matched_terms"
        ],
    )

    print(
        item[
            "source"
        ]
    )

{
  "source_notebook": "notebooks/20_dinov2_efficiency_frontier.ipynb",
  "matched_code_cells": 9,
  "explicit_l2_pattern_matches": [
    {
      "cell_index": 5,
      "patterns": [
        "torch.nn.functional.normalize"
      ]
    },
    {
      "cell_index": 6,
      "patterns": [
        "/ np.linalg.norm"
      ]
    },
    {
      "cell_index": 10,
      "patterns": [
        "/ np.linalg.norm"
      ]
    },
    {
      "cell_index": 12,
      "patterns": [
        "/ np.linalg.norm"
      ]
    }
  ],
  "archive_norm_min": 0.9999998807907104,
  "archive_norm_mean": 1.0,
  "archive_norm_max": 1.0000001192092896,
  "all_archive_features_unit_norm": true,
  "source_normalization_status": "explicit_normalization_found",
  "training_performed": false,
  "selection_evaluated": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false
}

Relevant Notebook 20 code cells:

Cell index: 4
Matched terms: ['dinov2_vits14_reg', 'vits14_reg']
torch.cuda.empty

In [5]:
STUDENT_SEED = 42

DINO_S_FEATURE_DIM = 384
DINO_S_PROJECTION_DIM = 144

DINO_S_FEATURES_PRENORMALIZED = True
DINO_S_PROJECTION_BIAS = True

DINO_S_STUDENT_EPOCHS = 10

DINO_S_PROJECTION_LR = 1e-3
DINO_S_WEIGHT_DECAY = 1e-4
DINO_S_GRAD_CLIP_MAX_NORM = 5.0

DINO_S_COSINE_MARGIN = 0.5

WGC_REFERENCE_CROSS_MACRO_AUC = (
    0.7827656525573192
)

DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC = (
    0.8430555555555557
)

STUDENT_CROSS_CONDITIONS = [
    (
        "cross_variable_variable",
        0,
        2,
    ),
    (
        "cross_variable_same",
        0,
        3,
    ),
    (
        "cross_same_variable",
        1,
        2,
    ),
    (
        "cross_same_same",
        1,
        3,
    ),
]


def set_student_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


class DINOv2SProjectionStudent(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.projection = nn.Linear(
            DINO_S_FEATURE_DIM,
            DINO_S_PROJECTION_DIM,
            bias=DINO_S_PROJECTION_BIAS,
        )

        nn.init.xavier_uniform_(
            self.projection.weight
        )

        if (
            self.projection.bias
            is not None
        ):
            nn.init.zeros_(
                self.projection.bias
            )

    def forward(
        self,
        features,
    ):
        normalized_features = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        projected_features = (
            self.projection(
                normalized_features
            )
        )

        embeddings = F.normalize(
            projected_features,
            p=2,
            dim=-1,
        )

        return embeddings


def build_dinov2s_projection_student(
    seed,
):
    set_student_seed(
        seed
    )

    return (
        DINOv2SProjectionStudent()
    )


def build_dinov2s_projection_optimizer(
    model,
):
    return torch.optim.AdamW(
        model.parameters(),
        lr=DINO_S_PROJECTION_LR,
        weight_decay=DINO_S_WEIGHT_DECAY,
    )


def balanced_writer_metric_loss(
    left_embeddings,
    right_embeddings,
):
    similarity = (
        left_embeddings
        @ right_embeddings.T
    )

    batch_size = int(
        similarity.shape[
            0
        ]
    )

    diagonal_mask = torch.eye(
        batch_size,
        dtype=torch.bool,
        device=similarity.device,
    )

    positive_scores = (
        similarity[
            diagonal_mask
        ]
    )

    negative_scores = (
        similarity[
            ~diagonal_mask
        ]
    )

    positive_loss = (
        1.0
        - positive_scores
    ).mean()

    negative_loss = torch.clamp(
        negative_scores
        - DINO_S_COSINE_MARGIN,
        min=0.0,
    ).mean()

    loss = (
        positive_loss
        + negative_loss
    ) / 2.0

    return {
        "loss": (
            loss
        ),
        "positive_loss": (
            positive_loss
        ),
        "negative_loss": (
            negative_loss
        ),
        "positive_similarity_mean": (
            positive_scores.mean()
        ),
        "negative_similarity_mean": (
            negative_scores.mean()
        ),
    }


def build_dinov2s_base_objective(
    page_embeddings,
):
    arabic_result = (
        balanced_writer_metric_loss(
            page_embeddings[
                :,
                0,
                :,
            ],
            page_embeddings[
                :,
                1,
                :,
            ],
        )
    )

    cross_results = {}

    cross_losses = []

    for (
        condition,
        left_page,
        right_page,
    ) in STUDENT_CROSS_CONDITIONS:
        condition_result = (
            balanced_writer_metric_loss(
                page_embeddings[
                    :,
                    left_page,
                    :,
                ],
                page_embeddings[
                    :,
                    right_page,
                    :,
                ],
            )
        )

        cross_results[
            condition
        ] = (
            condition_result
        )

        cross_losses.append(
            condition_result[
                "loss"
            ]
        )

    cross_metric_loss = (
        torch.stack(
            cross_losses
        ).mean()
    )

    base_loss = (
        arabic_result[
            "loss"
        ]
        + cross_metric_loss
    )

    return {
        "base_loss": (
            base_loss
        ),
        "arabic_result": (
            arabic_result
        ),
        "cross_metric_loss": (
            cross_metric_loss
        ),
        "cross_results": (
            cross_results
        ),
    }


protocol_model = (
    build_dinov2s_projection_student(
        STUDENT_SEED
    )
)

first_batch_feature_tensor = (
    torch.from_numpy(
        first_batch_raw_features
    )
    .float()
)

with torch.no_grad():
    first_batch_initial_embeddings = (
        protocol_model(
            first_batch_feature_tensor
        )
    )

initial_embedding_norms = (
    torch.linalg.vector_norm(
        first_batch_initial_embeddings,
        dim=-1,
    )
)

input_feature_norms = (
    torch.linalg.vector_norm(
        first_batch_feature_tensor,
        dim=-1,
    )
)

total_projection_parameters = int(
    sum(
        parameter.numel()
        for parameter in (
            protocol_model.parameters()
        )
    )
)

trainable_projection_parameters = int(
    sum(
        parameter.numel()
        for parameter in (
            protocol_model.parameters()
        )
        if parameter.requires_grad
    )
)

dinov2s_student_protocol = {
    "student": (
        "DINOv2-S frozen feature + "
        "trainable linear projection"
    ),
    "feature_source": (
        "cached DINOv2-S/14-Reg CLS embeddings"
    ),
    "feature_dimension": int(
        DINO_S_FEATURE_DIM
    ),
    "source_features_l2_normalized": True,
    "source_normalization_provenance": (
        "Notebook 20 extraction applied "
        "torch.nn.functional.normalize before archive"
    ),
    "input_normalization_preserved": True,
    "projection_dimension": int(
        DINO_S_PROJECTION_DIM
    ),
    "projection_bias": bool(
        DINO_S_PROJECTION_BIAS
    ),
    "projection_initialization": (
        "xavier_uniform_weight_zero_bias"
    ),
    "output_l2_normalization": True,
    "verification": "cosine similarity",
    "training_objective": (
        "matched Arabic plus four-condition "
        "cross-script writer metric objective"
    ),
    "auxiliary_loss": None,
    "wgc_used": False,
    "teacher_distillation_used": False,
    "script_adversarial_loss_used": False,
    "epochs": int(
        DINO_S_STUDENT_EPOCHS
    ),
    "learning_rate": float(
        DINO_S_PROJECTION_LR
    ),
    "optimizer": "AdamW",
    "weight_decay": float(
        DINO_S_WEIGHT_DECAY
    ),
    "gradient_clip_max_norm": float(
        DINO_S_GRAD_CLIP_MAX_NORM
    ),
    "cosine_margin": float(
        DINO_S_COSINE_MARGIN
    ),
    "training_seed": int(
        STUDENT_SEED
    ),
    "batch_schedule": (
        "Notebook 23 fixed writer-centric schedule"
    ),
    "checkpoint_selection_rule": (
        "cross_macro_auc > pooled_auc > "
        "lower_pooled_eer > earlier_epoch"
    ),
    "wgc_reference_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "pca_lda_reference_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "projection_parameters": int(
        total_projection_parameters
    ),
    "trainable_parameters": int(
        trainable_projection_parameters
    ),
    "first_batch_input_shape": list(
        first_batch_feature_tensor.shape
    ),
    "first_batch_output_shape": list(
        first_batch_initial_embeddings.shape
    ),
    "input_norm_min": float(
        input_feature_norms.min()
    ),
    "input_norm_max": float(
        input_feature_norms.max()
    ),
    "output_norm_min": float(
        initial_embedding_norms.min()
    ),
    "output_norm_max": float(
        initial_embedding_norms.max()
    ),
    "architecture_frozen_before_training": True,
    "optimizer_protocol_frozen_before_training": True,
    "selection_used_for_hyperparameter_tuning": False,
    "training_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_student_protocol.json",
    "w",
) as file:
    json.dump(
        dinov2s_student_protocol,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_student_protocol,
        indent=2,
    )
)

if (
    total_projection_parameters
    != 55440
    or trainable_projection_parameters
    != 55440
    or first_batch_feature_tensor.shape
    != (
        8,
        4,
        384,
    )
    or first_batch_initial_embeddings.shape
    != (
        8,
        4,
        144,
    )
    or not torch.allclose(
        input_feature_norms,
        torch.ones_like(
            input_feature_norms
        ),
        atol=1e-5,
    )
    or not torch.allclose(
        initial_embedding_norms,
        torch.ones_like(
            initial_embedding_norms
        ),
        atol=1e-5,
    )
):
    raise RuntimeError(
        "DINOv2-S projection student protocol audit failed."
    )

del protocol_model

{
  "student": "DINOv2-S frozen feature + trainable linear projection",
  "feature_source": "cached DINOv2-S/14-Reg CLS embeddings",
  "feature_dimension": 384,
  "source_features_l2_normalized": true,
  "source_normalization_provenance": "Notebook 20 extraction applied torch.nn.functional.normalize before archive",
  "input_normalization_preserved": true,
  "projection_dimension": 144,
  "projection_bias": true,
  "projection_initialization": "xavier_uniform_weight_zero_bias",
  "output_l2_normalization": true,
  "verification": "cosine similarity",
  "training_objective": "matched Arabic plus four-condition cross-script writer metric objective",
  "auxiliary_loss": null,
  "wgc_used": false,
  "teacher_distillation_used": false,
  "script_adversarial_loss_used": false,
  "epochs": 10,
  "learning_rate": 0.001,
  "optimizer": "AdamW",
  "weight_decay": 0.0001,
  "gradient_clip_max_norm": 5.0,
  "cosine_margin": 0.5,
  "training_seed": 42,
  "batch_schedule": "Notebook 23 fixed writer-

In [6]:
gradient_sanity_model = (
    build_dinov2s_projection_student(
        STUDENT_SEED
    )
)

gradient_sanity_model.train()

batch_features = (
    torch.from_numpy(
        first_batch_raw_features
    )
    .float()
)

initial_parameter_copies = {
    name: parameter
    .detach()
    .clone()
    for name, parameter in (
        gradient_sanity_model
        .named_parameters()
    )
}

gradient_sanity_model.zero_grad(
    set_to_none=True
)

batch_embeddings = (
    gradient_sanity_model(
        batch_features
    )
)

base_objective = (
    build_dinov2s_base_objective(
        batch_embeddings
    )
)

base_loss = (
    base_objective[
        "base_loss"
    ]
)

base_loss.backward()

gradient_rows = []

total_gradient_squared_sum = 0.0

for name, parameter in (
    gradient_sanity_model
    .named_parameters()
):
    if parameter.grad is None:
        gradient_norm = None
        gradient_finite = False
    else:
        gradient_norm = float(
            parameter.grad
            .detach()
            .double()
            .norm()
            .item()
        )

        gradient_finite = bool(
            torch.isfinite(
                parameter.grad
            ).all()
        )

        total_gradient_squared_sum += float(
            parameter.grad
            .detach()
            .double()
            .pow(
                2
            )
            .sum()
            .item()
        )

    parameter_change = (
        parameter
        .detach()
        - initial_parameter_copies[
            name
        ]
    )

    gradient_rows.append(
        {
            "parameter": (
                name
            ),
            "shape": str(
                tuple(
                    parameter.shape
                )
            ),
            "requires_grad": bool(
                parameter.requires_grad
            ),
            "gradient_present": bool(
                parameter.grad
                is not None
            ),
            "gradient_finite": bool(
                gradient_finite
            ),
            "gradient_norm": (
                gradient_norm
            ),
            "parameter_change_after_backward": float(
                parameter_change
                .abs()
                .max()
                .item()
            ),
        }
    )

gradient_sanity_df = pd.DataFrame(
    gradient_rows
)

total_gradient_norm = float(
    np.sqrt(
        total_gradient_squared_sum
    )
)

condition_rows = []

for condition, result in (
    base_objective[
        "cross_results"
    ].items()
):
    condition_rows.append(
        {
            "condition": (
                condition
            ),
            "loss": float(
                result[
                    "loss"
                ]
                .detach()
                .item()
            ),
            "positive_loss": float(
                result[
                    "positive_loss"
                ]
                .detach()
                .item()
            ),
            "negative_loss": float(
                result[
                    "negative_loss"
                ]
                .detach()
                .item()
            ),
            "positive_similarity_mean": float(
                result[
                    "positive_similarity_mean"
                ]
                .detach()
                .item()
            ),
            "negative_similarity_mean": float(
                result[
                    "negative_similarity_mean"
                ]
                .detach()
                .item()
            ),
        }
    )

condition_sanity_df = pd.DataFrame(
    condition_rows
)

output_norms = (
    torch.linalg.vector_norm(
        batch_embeddings,
        dim=-1,
    )
)

all_gradients_present = bool(
    gradient_sanity_df[
        "gradient_present"
    ].all()
)

all_gradients_finite = bool(
    gradient_sanity_df[
        "gradient_finite"
    ].all()
)

maximum_parameter_change = float(
    gradient_sanity_df[
        "parameter_change_after_backward"
    ].max()
)

dinov2s_projection_gradient_sanity = {
    "batch_writers": (
        first_batch_writers
    ),
    "batch_size": int(
        batch_features.shape[
            0
        ]
    ),
    "input_shape": list(
        batch_features.shape
    ),
    "embedding_shape": list(
        batch_embeddings.shape
    ),
    "base_loss": float(
        base_loss
        .detach()
        .item()
    ),
    "arabic_metric_loss": float(
        base_objective[
            "arabic_result"
        ][
            "loss"
        ]
        .detach()
        .item()
    ),
    "cross_metric_loss": float(
        base_objective[
            "cross_metric_loss"
        ]
        .detach()
        .item()
    ),
    "output_norm_min": float(
        output_norms.min()
        .detach()
        .item()
    ),
    "output_norm_max": float(
        output_norms.max()
        .detach()
        .item()
    ),
    "trainable_parameter_tensors": int(
        len(
            gradient_sanity_df
        )
    ),
    "trainable_parameters": int(
        sum(
            parameter.numel()
            for parameter in (
                gradient_sanity_model
                .parameters()
            )
            if parameter.requires_grad
        )
    ),
    "all_gradients_present": bool(
        all_gradients_present
    ),
    "all_gradients_finite": bool(
        all_gradients_finite
    ),
    "total_gradient_norm": float(
        total_gradient_norm
    ),
    "maximum_parameter_change_after_backward": float(
        maximum_parameter_change
    ),
    "optimizer_created": False,
    "optimizer_step_performed": False,
    "parameter_update_performed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

gradient_sanity_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_gradient_parameter_audit.csv",
    index=False,
)

condition_sanity_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_initial_condition_sanity.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_gradient_sanity.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_gradient_sanity,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_gradient_sanity,
        indent=2,
    )
)

print(
    "\nGradient audit:"
)

print(
    gradient_sanity_df
    .round(
        8
    )
    .to_string(
        index=False
    )
)

print(
    "\nCondition sanity:"
)

print(
    condition_sanity_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    batch_features.shape
    != (
        8,
        4,
        384,
    )
    or batch_embeddings.shape
    != (
        8,
        4,
        144,
    )
    or not torch.isfinite(
        base_loss
    )
    or base_loss.item()
    <= 0.0
    or not torch.allclose(
        output_norms,
        torch.ones_like(
            output_norms
        ),
        atol=1e-5,
    )
    or not all_gradients_present
    or not all_gradients_finite
    or total_gradient_norm
    <= 0.0
    or maximum_parameter_change
    != 0.0
):
    raise RuntimeError(
        "DINOv2-S projection gradient sanity failed."
    )

{
  "batch_writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "batch_size": 8,
  "input_shape": [
    8,
    4,
    384
  ],
  "embedding_shape": [
    8,
    4,
    144
  ],
  "base_loss": 0.45000821352005005,
  "arabic_metric_loss": 0.21952931582927704,
  "cross_metric_loss": 0.2304789125919342,
  "output_norm_min": 0.9999998807907104,
  "output_norm_max": 1.0,
  "trainable_parameter_tensors": 2,
  "trainable_parameters": 55440,
  "all_gradients_present": true,
  "all_gradients_finite": true,
  "total_gradient_norm": 0.18966344137204308,
  "maximum_parameter_change_after_backward": 0.0,
  "optimizer_created": false,
  "optimizer_step_performed": false,
  "parameter_update_performed": false,
  "selection_evaluated": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false
}

Gradient audit:
        parameter      shape  requires_grad  gradient_present  gradient_finite  gradient_norm  parameter_change_after_backward


In [7]:
optimizer_sanity_model = (
    build_dinov2s_projection_student(
        STUDENT_SEED
    )
)

optimizer_sanity_optimizer = (
    build_dinov2s_projection_optimizer(
        optimizer_sanity_model
    )
)

optimizer_sanity_model.train()

optimizer_batch_features = (
    torch.from_numpy(
        first_batch_raw_features
    )
    .float()
)

initial_parameters = {
    name: parameter
    .detach()
    .clone()
    for name, parameter in (
        optimizer_sanity_model
        .named_parameters()
    )
}

optimizer_sanity_optimizer.zero_grad(
    set_to_none=True
)

pre_step_embeddings = (
    optimizer_sanity_model(
        optimizer_batch_features
    )
)

pre_step_objective = (
    build_dinov2s_base_objective(
        pre_step_embeddings
    )
)

pre_step_loss = (
    pre_step_objective[
        "base_loss"
    ]
)

pre_step_loss.backward()

trainable_parameters = [
    parameter
    for parameter in (
        optimizer_sanity_model
        .parameters()
    )
    if parameter.requires_grad
]

gradient_norm_before_clip = (
    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=(
            DINO_S_GRAD_CLIP_MAX_NORM
        ),
    )
)

optimizer_sanity_optimizer.step()

parameter_update_rows = []

total_update_squared_sum = 0.0

for name, parameter in (
    optimizer_sanity_model
    .named_parameters()
):
    update = (
        parameter
        .detach()
        - initial_parameters[
            name
        ]
    )

    update_norm = float(
        update
        .double()
        .norm()
        .item()
    )

    update_max = float(
        update
        .abs()
        .max()
        .item()
    )

    total_update_squared_sum += float(
        update
        .double()
        .pow(
            2
        )
        .sum()
        .item()
    )

    parameter_update_rows.append(
        {
            "parameter": (
                name
            ),
            "shape": str(
                tuple(
                    parameter.shape
                )
            ),
            "update_norm": float(
                update_norm
            ),
            "maximum_absolute_update": float(
                update_max
            ),
            "changed": bool(
                update_max
                > 0.0
            ),
        }
    )

parameter_update_df = pd.DataFrame(
    parameter_update_rows
)

with torch.no_grad():
    post_step_embeddings = (
        optimizer_sanity_model(
            optimizer_batch_features
        )
    )

    post_step_objective = (
        build_dinov2s_base_objective(
            post_step_embeddings
        )
    )

post_step_loss = float(
    post_step_objective[
        "base_loss"
    ].item()
)

total_parameter_update_norm = float(
    np.sqrt(
        total_update_squared_sum
    )
)

post_step_norms = (
    torch.linalg.vector_norm(
        post_step_embeddings,
        dim=-1,
    )
)

dinov2s_optimizer_step_sanity = {
    "batch_writers": (
        first_batch_writers
    ),
    "batch_size": int(
        optimizer_batch_features.shape[
            0
        ]
    ),
    "learning_rate": float(
        DINO_S_PROJECTION_LR
    ),
    "weight_decay": float(
        DINO_S_WEIGHT_DECAY
    ),
    "pre_step_base_loss": float(
        pre_step_loss
        .detach()
        .item()
    ),
    "post_step_base_loss_same_batch": float(
        post_step_loss
    ),
    "same_batch_loss_change": float(
        post_step_loss
        - pre_step_loss
        .detach()
        .item()
    ),
    "gradient_norm_before_clip": float(
        gradient_norm_before_clip
        .detach()
        .item()
    ),
    "gradient_clip_threshold": float(
        DINO_S_GRAD_CLIP_MAX_NORM
    ),
    "gradient_clipping_activated": bool(
        gradient_norm_before_clip
        .detach()
        .item()
        > DINO_S_GRAD_CLIP_MAX_NORM
    ),
    "parameter_tensors": int(
        len(
            parameter_update_df
        )
    ),
    "changed_parameter_tensors": int(
        parameter_update_df[
            "changed"
        ].sum()
    ),
    "total_parameter_update_norm": float(
        total_parameter_update_norm
    ),
    "maximum_absolute_update": float(
        parameter_update_df[
            "maximum_absolute_update"
        ].max()
    ),
    "post_step_output_norm_min": float(
        post_step_norms.min()
        .item()
    ),
    "post_step_output_norm_max": float(
        post_step_norms.max()
        .item()
    ),
    "optimizer_step_performed": True,
    "training_experiment_started": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

parameter_update_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_optimizer_step_parameter_audit.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_optimizer_step_sanity.json",
    "w",
) as file:
    json.dump(
        dinov2s_optimizer_step_sanity,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_optimizer_step_sanity,
        indent=2,
    )
)

print(
    "\nParameter updates:"
)

print(
    parameter_update_df
    .round(
        8
    )
    .to_string(
        index=False
    )
)

if (
    not np.isfinite(
        dinov2s_optimizer_step_sanity[
            "pre_step_base_loss"
        ]
    )
    or not np.isfinite(
        dinov2s_optimizer_step_sanity[
            "post_step_base_loss_same_batch"
        ]
    )
    or dinov2s_optimizer_step_sanity[
        "gradient_norm_before_clip"
    ]
    <= 0.0
    or dinov2s_optimizer_step_sanity[
        "changed_parameter_tensors"
    ]
    != 2
    or dinov2s_optimizer_step_sanity[
        "total_parameter_update_norm"
    ]
    <= 0.0
    or dinov2s_optimizer_step_sanity[
        "maximum_absolute_update"
    ]
    <= 0.0
    or not torch.allclose(
        post_step_norms,
        torch.ones_like(
            post_step_norms
        ),
        atol=1e-5,
    )
):
    raise RuntimeError(
        "DINOv2-S projection optimizer-step sanity failed."
    )

del optimizer_sanity_model
del optimizer_sanity_optimizer
del initial_parameters
del optimizer_batch_features
del pre_step_embeddings
del pre_step_objective
del pre_step_loss
del post_step_embeddings
del post_step_objective

{
  "batch_writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "batch_size": 8,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "pre_step_base_loss": 0.45000821352005005,
  "post_step_base_loss_same_batch": 0.41752171516418457,
  "same_batch_loss_change": -0.03248649835586548,
  "gradient_norm_before_clip": 0.18966343998908997,
  "gradient_clip_threshold": 5.0,
  "gradient_clipping_activated": false,
  "parameter_tensors": 2,
  "changed_parameter_tensors": 2,
  "total_parameter_update_norm": 0.23541354370403628,
  "maximum_absolute_update": 0.0010000094771385193,
  "post_step_output_norm_min": 0.9999999403953552,
  "post_step_output_norm_max": 1.0,
  "optimizer_step_performed": true,
  "training_experiment_started": false,
  "selection_evaluated": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false
}

Parameter updates:
        parameter      shape  update_norm  maximum_absolute_update  changed
projection.w

In [9]:
SELECTION_PAIR_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "selection_cross_script_pairs.csv"
)

NOTEBOOK26_ADAPTED_PATH = (
    ROOT
    / "reports"
    / "dinov2s_clean_representation_audit"
    / "dinov2s_fit_selection_adapted_embeddings.npz"
)

EXPECTED_DINOV2S_PCA_LDA_MACRO_AUC = (
    0.8430555555555557
)

EXPECTED_DINOV2S_PCA_LDA_POOLED_AUC = (
    0.8429232804232805
)

EXPECTED_DINOV2S_PCA_LDA_POOLED_EER = (
    0.22916666666666663
)

selection_pair_df = pd.read_csv(
    SELECTION_PAIR_PATH
)

required_pair_columns = {
    "condition",
    "left_filename",
    "right_filename",
    "label",
}

if not required_pair_columns.issubset(
    selection_pair_df.columns
):
    raise RuntimeError(
        "Unexpected selection-pair schema."
    )


def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = (
        1.0
        - tpr
    )

    difference = (
        fpr
        - fnr
    )

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        )
        != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = int(
            np.argmin(
                np.abs(
                    difference
                )
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = (
            thresholds[
                nearest_index
            ]
        )

        return (
            float(
                eer
            ),
            float(
                threshold
            ),
        )

    index = int(
        crossing_indices[
            0
        ]
    )

    difference_left = (
        difference[
            index
        ]
    )

    difference_right = (
        difference[
            index
            + 1
        ]
    )

    interpolation_weight = (
        -difference_left
        / (
            difference_right
            - difference_left
        )
    )

    eer = (
        fpr[
            index
        ]
        + interpolation_weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + interpolation_weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return (
        float(
            eer
        ),
        float(
            threshold
        ),
    )


def evaluate_selection_embedding_map(
    embedding_map,
):
    missing_left = [
        filename
        for filename in (
            selection_pair_df[
                "left_filename"
            ].astype(str)
        )
        if filename
        not in embedding_map
    ]

    missing_right = [
        filename
        for filename in (
            selection_pair_df[
                "right_filename"
            ].astype(str)
        )
        if filename
        not in embedding_map
    ]

    if (
        len(
            missing_left
        ) > 0
        or len(
            missing_right
        ) > 0
    ):
        raise RuntimeError(
            "Selection filenames missing from embedding map."
        )

    left_embeddings = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                selection_pair_df[
                    "left_filename"
                ].astype(str)
            )
        ]
    )

    right_embeddings = np.stack(
        [
            embedding_map[
                filename
            ]
            for filename in (
                selection_pair_df[
                    "right_filename"
                ].astype(str)
            )
        ]
    )

    labels = (
        selection_pair_df[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    scores = np.sum(
        left_embeddings
        * right_embeddings,
        axis=1,
    )

    pooled_auc = float(
        roc_auc_score(
            labels,
            scores,
        )
    )

    pooled_eer, pooled_eer_threshold = (
        calculate_interpolated_eer(
            labels,
            scores,
        )
    )

    scored_pairs = (
        selection_pair_df
        .copy()
    )

    scored_pairs[
        "score"
    ] = scores

    condition_rows = []

    for condition in sorted(
        scored_pairs[
            "condition"
        ].unique()
    ):
        condition_df = (
            scored_pairs[
                scored_pairs[
                    "condition"
                ] == condition
            ]
        )

        condition_labels = (
            condition_df[
                "label"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        condition_scores = (
            condition_df[
                "score"
            ]
            .to_numpy()
        )

        condition_auc = float(
            roc_auc_score(
                condition_labels,
                condition_scores,
            )
        )

        condition_eer, _ = (
            calculate_interpolated_eer(
                condition_labels,
                condition_scores,
            )
        )

        condition_rows.append(
            {
                "condition": (
                    condition
                ),
                "auc": float(
                    condition_auc
                ),
                "eer": float(
                    condition_eer
                ),
                "pairs": int(
                    len(
                        condition_df
                    )
                ),
                "genuine_pairs": int(
                    condition_df[
                        "label"
                    ].sum()
                ),
                "impostor_pairs": int(
                    (
                        condition_df[
                            "label"
                        ] == 0
                    ).sum()
                ),
            }
        )

    condition_df = pd.DataFrame(
        condition_rows
    )

    return {
        "cross_macro_auc": float(
            condition_df[
                "auc"
            ].mean()
        ),
        "cross_macro_eer": float(
            condition_df[
                "eer"
            ].mean()
        ),
        "pooled_auc": float(
            pooled_auc
        ),
        "pooled_eer": float(
            pooled_eer
        ),
        "pooled_eer_threshold": float(
            pooled_eer_threshold
        ),
        "condition_df": (
            condition_df
        ),
        "missing_left_filenames": int(
            len(
                missing_left
            )
        ),
        "missing_right_filenames": int(
            len(
                missing_right
            )
        ),
    }


def evaluate_projection_student_selection(
    model,
):
    model.eval()

    model_device = next(
        model.parameters()
    ).device

    selection_tensor = (
        torch.from_numpy(
            selection_raw_features
        )
        .float()
        .to(
            model_device
        )
    )

    with torch.no_grad():
        embeddings = (
            model(
                selection_tensor
            )
            .detach()
            .cpu()
            .numpy()
        )

    embedding_map = {
        filename: embedding
        for filename, embedding in zip(
            selection_rows[
                "filename"
            ].astype(str),
            embeddings,
        )
    }

    return (
        evaluate_selection_embedding_map(
            embedding_map
        )
    )


notebook26_adapted = np.load(
    NOTEBOOK26_ADAPTED_PATH,
    allow_pickle=True,
)

required_reference_keys = {
    "selection_embeddings",
    "selection_filenames",
    "selection_writers",
    "selection_page_ids",
}

if not required_reference_keys.issubset(
    notebook26_adapted.files
):
    raise RuntimeError(
        "Notebook 26 adapted reference artifact is incomplete."
    )

reference_selection_embeddings = (
    notebook26_adapted[
        "selection_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

reference_selection_filenames = (
    notebook26_adapted[
        "selection_filenames"
    ]
    .astype(str)
)

reference_selection_writers = (
    notebook26_adapted[
        "selection_writers"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

reference_selection_page_ids = (
    notebook26_adapted[
        "selection_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

reference_embedding_map = {
    filename: embedding
    for filename, embedding in zip(
        reference_selection_filenames,
        reference_selection_embeddings,
    )
}

reference_result = (
    evaluate_selection_embedding_map(
        reference_embedding_map
    )
)

macro_difference = float(
    reference_result[
        "cross_macro_auc"
    ]
    - EXPECTED_DINOV2S_PCA_LDA_MACRO_AUC
)

pooled_auc_difference = float(
    reference_result[
        "pooled_auc"
    ]
    - EXPECTED_DINOV2S_PCA_LDA_POOLED_AUC
)

pooled_eer_difference = float(
    reference_result[
        "pooled_eer"
    ]
    - EXPECTED_DINOV2S_PCA_LDA_POOLED_EER
)

reference_condition_df = (
    reference_result[
        "condition_df"
    ]
)

dinov2s_selection_evaluator_audit = {
    "reference_artifact": str(
        NOTEBOOK26_ADAPTED_PATH.relative_to(
            ROOT
        )
    ),
    "reference_selection_embedding_shape": list(
        reference_selection_embeddings.shape
    ),
    "reference_selection_filenames": int(
        len(
            reference_selection_filenames
        )
    ),
    "reference_selection_writers": int(
        len(
            np.unique(
                reference_selection_writers
            )
        )
    ),
    "reference_page_ids": sorted(
        np.unique(
            reference_selection_page_ids
        )
        .astype(int)
        .tolist()
    ),
    "selection_pairs": int(
        len(
            selection_pair_df
        )
    ),
    "selection_conditions": int(
        selection_pair_df[
            "condition"
        ].nunique()
    ),
    "expected_cross_macro_auc": float(
        EXPECTED_DINOV2S_PCA_LDA_MACRO_AUC
    ),
    "reproduced_cross_macro_auc": float(
        reference_result[
            "cross_macro_auc"
        ]
    ),
    "cross_macro_auc_difference": float(
        macro_difference
    ),
    "expected_pooled_auc": float(
        EXPECTED_DINOV2S_PCA_LDA_POOLED_AUC
    ),
    "reproduced_pooled_auc": float(
        reference_result[
            "pooled_auc"
        ]
    ),
    "pooled_auc_difference": float(
        pooled_auc_difference
    ),
    "expected_pooled_eer": float(
        EXPECTED_DINOV2S_PCA_LDA_POOLED_EER
    ),
    "reproduced_pooled_eer": float(
        reference_result[
            "pooled_eer"
        ]
    ),
    "pooled_eer_difference": float(
        pooled_eer_difference
    ),
    "condition_pair_counts_valid": bool(
        (
            reference_condition_df[
                "pairs"
            ]
            == 1296
        ).all()
    ),
    "condition_genuine_counts_valid": bool(
        (
            reference_condition_df[
                "genuine_pairs"
            ]
            == 36
        ).all()
    ),
    "condition_impostor_counts_valid": bool(
        (
            reference_condition_df[
                "impostor_pairs"
            ]
            == 1260
        ).all()
    ),
    "missing_left_filenames": int(
        reference_result[
            "missing_left_filenames"
        ]
    ),
    "missing_right_filenames": int(
        reference_result[
            "missing_right_filenames"
        ]
    ),
    "evaluator_frozen": True,
    "projection_student_evaluated": False,
    "projection_training_performed": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

reference_condition_df.to_csv(
    REPORT_DIR
    / "notebook26_reference_reproduction_conditions.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_selection_evaluator_audit.json",
    "w",
) as file:
    json.dump(
        dinov2s_selection_evaluator_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_selection_evaluator_audit,
        indent=2,
    )
)

print(
    "\nNotebook 26 condition reproduction:"
)

print(
    reference_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    reference_selection_embeddings.shape
    != (
        144,
        144,
    )
    or len(
        reference_selection_filenames
    ) != 144
    or len(
        np.unique(
            reference_selection_writers
        )
    ) != 36
    or len(
        selection_pair_df
    ) != 5184
    or selection_pair_df[
        "condition"
    ].nunique()
    != 4
    or abs(
        macro_difference
    ) > 1e-12
    or abs(
        pooled_auc_difference
    ) > 1e-12
    or abs(
        pooled_eer_difference
    ) > 1e-12
    or not (
        reference_condition_df[
            "pairs"
        ]
        == 1296
    ).all()
    or not (
        reference_condition_df[
            "genuine_pairs"
        ]
        == 36
    ).all()
    or not (
        reference_condition_df[
            "impostor_pairs"
        ]
        == 1260
    ).all()
):
    raise RuntimeError(
        "DINOv2-S selection evaluator failed reference reproduction."
    )

{
  "reference_artifact": "reports/dinov2s_clean_representation_audit/dinov2s_fit_selection_adapted_embeddings.npz",
  "reference_selection_embedding_shape": [
    144,
    144
  ],
  "reference_selection_filenames": 144,
  "reference_selection_writers": 36,
  "reference_page_ids": [
    1,
    2,
    3,
    4
  ],
  "selection_pairs": 5184,
  "selection_conditions": 4,
  "expected_cross_macro_auc": 0.8430555555555557,
  "reproduced_cross_macro_auc": 0.8430555555555557,
  "cross_macro_auc_difference": 0.0,
  "expected_pooled_auc": 0.8429232804232805,
  "reproduced_pooled_auc": 0.8429232804232805,
  "pooled_auc_difference": 0.0,
  "expected_pooled_eer": 0.22916666666666663,
  "reproduced_pooled_eer": 0.22916666666666663,
  "pooled_eer_difference": 0.0,
  "condition_pair_counts_valid": true,
  "condition_genuine_counts_valid": true,
  "condition_impostor_counts_valid": true,
  "missing_left_filenames": 0,
  "missing_right_filenames": 0,
  "evaluator_frozen": true,
  "projection_student_e

In [10]:
STUDENT_DEVICE = torch.device(
    "cpu"
)


def load_projection_writer_batch(
    epoch,
    batch_id,
):
    batch_schedule = (
        fit_schedule_df[
            (
                fit_schedule_df[
                    "epoch"
                ] == epoch
            )
            & (
                fit_schedule_df[
                    "batch"
                ] == batch_id
            )
        ]
        .sort_values(
            "slot"
        )
    )

    writers = (
        batch_schedule[
            "writer"
        ]
        .astype(int)
        .tolist()
    )

    features = np.stack(
        [
            fit_writer_feature_map[
                writer
            ]
            for writer in (
                writers
            )
        ],
        axis=0,
    )

    return {
        "writers": (
            writers
        ),
        "features": (
            features
            .astype(
                np.float32,
                copy=False,
            )
        ),
    }


def train_projection_batch(
    model,
    optimizer,
    epoch,
    batch_id,
):
    model.train()

    batch = (
        load_projection_writer_batch(
            epoch=epoch,
            batch_id=batch_id,
        )
    )

    writers = (
        batch[
            "writers"
        ]
    )

    features = (
        torch.from_numpy(
            batch[
                "features"
            ]
        )
        .float()
        .to(
            STUDENT_DEVICE
        )
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    embeddings = (
        model(
            features
        )
    )

    objective = (
        build_dinov2s_base_objective(
            embeddings
        )
    )

    loss = (
        objective[
            "base_loss"
        ]
    )

    loss.backward()

    gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=(
                DINO_S_GRAD_CLIP_MAX_NORM
            ),
        )
    )

    optimizer.step()

    condition_metrics = {}

    for condition, result in (
        objective[
            "cross_results"
        ].items()
    ):
        condition_metrics[
            condition
        ] = {
            "loss": float(
                result[
                    "loss"
                ]
                .detach()
                .cpu()
                .item()
            ),
            "positive_similarity_mean": float(
                result[
                    "positive_similarity_mean"
                ]
                .detach()
                .cpu()
                .item()
            ),
            "negative_similarity_mean": float(
                result[
                    "negative_similarity_mean"
                ]
                .detach()
                .cpu()
                .item()
            ),
        }

    return {
        "epoch": int(
            epoch
        ),
        "batch": int(
            batch_id
        ),
        "batch_size": int(
            features.shape[
                0
            ]
        ),
        "writers": (
            writers
        ),
        "base_loss": float(
            loss
            .detach()
            .cpu()
            .item()
        ),
        "arabic_metric_loss": float(
            objective[
                "arabic_result"
            ][
                "loss"
            ]
            .detach()
            .cpu()
            .item()
        ),
        "cross_metric_loss": float(
            objective[
                "cross_metric_loss"
            ]
            .detach()
            .cpu()
            .item()
        ),
        "gradient_norm_before_clip": float(
            gradient_norm
            .detach()
            .cpu()
            .item()
        ),
        "gradient_clipping_activated": bool(
            gradient_norm
            .detach()
            .cpu()
            .item()
            > DINO_S_GRAD_CLIP_MAX_NORM
        ),
        "conditions": (
            condition_metrics
        ),
    }


def train_projection_epoch(
    model,
    optimizer,
    epoch,
):
    epoch_batch_ids = (
        fit_schedule_df[
            fit_schedule_df[
                "epoch"
            ] == epoch
        ][
            "batch"
        ]
        .drop_duplicates()
        .sort_values()
        .astype(int)
        .tolist()
    )

    batch_rows = []

    for batch_id in (
        epoch_batch_ids
    ):
        batch_result = (
            train_projection_batch(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                batch_id=batch_id,
            )
        )

        batch_rows.append(
            batch_result
        )

    batch_df = pd.DataFrame(
        [
            {
                key: value
                for key, value in (
                    row.items()
                )
                if key
                not in {
                    "writers",
                    "conditions",
                }
            }
            for row in (
                batch_rows
            )
        ]
    )

    epoch_summary = {
        "epoch": int(
            epoch
        ),
        "batches": int(
            len(
                batch_rows
            )
        ),
        "writers_seen": int(
            sum(
                row[
                    "batch_size"
                ]
                for row in (
                    batch_rows
                )
            )
        ),
        "base_loss": float(
            batch_df[
                "base_loss"
            ].mean()
        ),
        "arabic_metric_loss": float(
            batch_df[
                "arabic_metric_loss"
            ].mean()
        ),
        "cross_metric_loss": float(
            batch_df[
                "cross_metric_loss"
            ].mean()
        ),
        "gradient_norm_mean": float(
            batch_df[
                "gradient_norm_before_clip"
            ].mean()
        ),
        "gradient_norm_max": float(
            batch_df[
                "gradient_norm_before_clip"
            ].max()
        ),
        "gradient_clipping_events": int(
            batch_df[
                "gradient_clipping_activated"
            ].sum()
        ),
    }

    return (
        epoch_summary,
        batch_df,
    )


dry_run_model = (
    build_dinov2s_projection_student(
        STUDENT_SEED
    )
    .to(
        STUDENT_DEVICE
    )
)

dry_run_optimizer = (
    build_dinov2s_projection_optimizer(
        dry_run_model
    )
)

dry_run_initial_parameters = {
    name: parameter
    .detach()
    .clone()
    for name, parameter in (
        dry_run_model
        .named_parameters()
    )
}

dry_run_result = (
    train_projection_batch(
        model=dry_run_model,
        optimizer=dry_run_optimizer,
        epoch=1,
        batch_id=1,
    )
)

parameter_update_squared_sum = 0.0

changed_parameter_tensors = 0

maximum_absolute_update = 0.0

for name, parameter in (
    dry_run_model
    .named_parameters()
):
    update = (
        parameter
        .detach()
        - dry_run_initial_parameters[
            name
        ]
    )

    parameter_update_squared_sum += float(
        update
        .double()
        .pow(
            2
        )
        .sum()
        .cpu()
        .item()
    )

    current_maximum_update = float(
        update
        .abs()
        .max()
        .cpu()
        .item()
    )

    maximum_absolute_update = max(
        maximum_absolute_update,
        current_maximum_update,
    )

    if current_maximum_update > 0.0:
        changed_parameter_tensors += 1

dry_run_parameter_update_norm = float(
    np.sqrt(
        parameter_update_squared_sum
    )
)

dry_run_condition_df = pd.DataFrame(
    [
        {
            "condition": condition,
            **values,
        }
        for condition, values in (
            dry_run_result[
                "conditions"
            ].items()
        )
    ]
)

dinov2s_projection_trainer_dry_run = {
    "device": str(
        STUDENT_DEVICE
    ),
    "epoch": int(
        dry_run_result[
            "epoch"
        ]
    ),
    "batch": int(
        dry_run_result[
            "batch"
        ]
    ),
    "batch_size": int(
        dry_run_result[
            "batch_size"
        ]
    ),
    "writers": (
        dry_run_result[
            "writers"
        ]
    ),
    "base_loss": float(
        dry_run_result[
            "base_loss"
        ]
    ),
    "arabic_metric_loss": float(
        dry_run_result[
            "arabic_metric_loss"
        ]
    ),
    "cross_metric_loss": float(
        dry_run_result[
            "cross_metric_loss"
        ]
    ),
    "gradient_norm_before_clip": float(
        dry_run_result[
            "gradient_norm_before_clip"
        ]
    ),
    "gradient_clipping_activated": bool(
        dry_run_result[
            "gradient_clipping_activated"
        ]
    ),
    "parameter_update_norm": float(
        dry_run_parameter_update_norm
    ),
    "maximum_absolute_update": float(
        maximum_absolute_update
    ),
    "changed_parameter_tensors": int(
        changed_parameter_tensors
    ),
    "parameter_tensors": int(
        len(
            dry_run_initial_parameters
        )
    ),
    "matches_previous_one_step_base_loss": bool(
        np.isclose(
            dry_run_result[
                "base_loss"
            ],
            0.45000821352005005,
            rtol=0.0,
            atol=1e-7,
        )
    ),
    "full_epoch_executed": False,
    "full_training_executed": False,
    "selection_evaluated": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
    "trainer_ready": True,
}

dry_run_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_trainer_dry_run_conditions.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_trainer_dry_run.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_trainer_dry_run,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_trainer_dry_run,
        indent=2,
    )
)

print(
    "\nCondition dry-run:"
)

print(
    dry_run_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    dinov2s_projection_trainer_dry_run[
        "batch_size"
    ] != 8
    or not dinov2s_projection_trainer_dry_run[
        "matches_previous_one_step_base_loss"
    ]
    or dinov2s_projection_trainer_dry_run[
        "gradient_norm_before_clip"
    ] <= 0.0
    or dinov2s_projection_trainer_dry_run[
        "parameter_update_norm"
    ] <= 0.0
    or dinov2s_projection_trainer_dry_run[
        "maximum_absolute_update"
    ] <= 0.0
    or dinov2s_projection_trainer_dry_run[
        "changed_parameter_tensors"
    ] != 2
):
    raise RuntimeError(
        "DINOv2-S projection trainer dry-run failed."
    )

del dry_run_model
del dry_run_optimizer
del dry_run_initial_parameters

{
  "device": "cpu",
  "epoch": 1,
  "batch": 1,
  "batch_size": 8,
  "writers": [
    133,
    39,
    273,
    138,
    172,
    249,
    272,
    94
  ],
  "base_loss": 0.45000821352005005,
  "arabic_metric_loss": 0.21952931582927704,
  "cross_metric_loss": 0.2304789125919342,
  "gradient_norm_before_clip": 0.18966343998908997,
  "gradient_clipping_activated": false,
  "parameter_update_norm": 0.23541354370403628,
  "maximum_absolute_update": 0.0010000094771385193,
  "changed_parameter_tensors": 2,
  "parameter_tensors": 2,
  "matches_previous_one_step_base_loss": true,
  "full_epoch_executed": false,
  "full_training_executed": false,
  "selection_evaluated": false,
  "monitor_used": false,
  "validation_used": false,
  "official_test_used": false,
  "trainer_ready": true
}

Condition dry-run:
              condition     loss  positive_similarity_mean  negative_similarity_mean
cross_variable_variable 0.224862                  0.930258                  0.879983
    cross_variable_sa

In [11]:
def projection_checkpoint_is_better(
    candidate,
    incumbent,
):
    if incumbent is None:
        return True

    tolerance = 1e-12

    if (
        candidate[
            "cross_macro_auc"
        ]
        > incumbent[
            "cross_macro_auc"
        ]
        + tolerance
    ):
        return True

    if not np.isclose(
        candidate[
            "cross_macro_auc"
        ],
        incumbent[
            "cross_macro_auc"
        ],
        rtol=0.0,
        atol=tolerance,
    ):
        return False

    if (
        candidate[
            "pooled_auc"
        ]
        > incumbent[
            "pooled_auc"
        ]
        + tolerance
    ):
        return True

    if not np.isclose(
        candidate[
            "pooled_auc"
        ],
        incumbent[
            "pooled_auc"
        ],
        rtol=0.0,
        atol=tolerance,
    ):
        return False

    if (
        candidate[
            "pooled_eer"
        ]
        < incumbent[
            "pooled_eer"
        ]
        - tolerance
    ):
        return True

    if not np.isclose(
        candidate[
            "pooled_eer"
        ],
        incumbent[
            "pooled_eer"
        ],
        rtol=0.0,
        atol=tolerance,
    ):
        return False

    return (
        candidate[
            "epoch"
        ]
        < incumbent[
            "epoch"
        ]
    )


def run_dinov2s_projection_experiment(
    training_seed,
):
    set_student_seed(
        training_seed
    )

    model = (
        build_dinov2s_projection_student(
            training_seed
        )
        .to(
            STUDENT_DEVICE
        )
    )

    optimizer = (
        build_dinov2s_projection_optimizer(
            model
        )
    )

    checkpoint_path = (
        CHECKPOINT_DIR
        / (
            f"dinov2s_projection_seed"
            f"{training_seed}_best.pt"
        )
    )

    history_rows = []

    best_record = None

    for epoch in range(
        1,
        DINO_S_STUDENT_EPOCHS
        + 1,
    ):
        epoch_summary, epoch_batch_df = (
            train_projection_epoch(
                model=model,
                optimizer=optimizer,
                epoch=epoch,
            )
        )

        if (
            epoch_summary[
                "batches"
            ] != 18
            or epoch_summary[
                "writers_seen"
            ] != 145
        ):
            raise RuntimeError(
                "Projection-student epoch schedule audit failed."
            )

        selection_result = (
            evaluate_projection_student_selection(
                model
            )
        )

        candidate_record = {
            "epoch": int(
                epoch
            ),
            "cross_macro_auc": float(
                selection_result[
                    "cross_macro_auc"
                ]
            ),
            "cross_macro_eer": float(
                selection_result[
                    "cross_macro_eer"
                ]
            ),
            "pooled_auc": float(
                selection_result[
                    "pooled_auc"
                ]
            ),
            "pooled_eer": float(
                selection_result[
                    "pooled_eer"
                ]
            ),
            "pooled_eer_threshold": float(
                selection_result[
                    "pooled_eer_threshold"
                ]
            ),
        }

        selected = (
            projection_checkpoint_is_better(
                candidate_record,
                best_record,
            )
        )

        if selected:
            best_record = dict(
                candidate_record
            )

            torch.save(
                {
                    "method": (
                        "DINOv2-S frozen-feature "
                        "projection student"
                    ),
                    "training_seed": int(
                        training_seed
                    ),
                    "epoch": int(
                        epoch
                    ),
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),
                    "projection_dimension": int(
                        DINO_S_PROJECTION_DIM
                    ),
                    "learning_rate": float(
                        DINO_S_PROJECTION_LR
                    ),
                    "weight_decay": float(
                        DINO_S_WEIGHT_DECAY
                    ),
                    "selection_cross_macro_auc": float(
                        selection_result[
                            "cross_macro_auc"
                        ]
                    ),
                    "selection_cross_macro_eer": float(
                        selection_result[
                            "cross_macro_eer"
                        ]
                    ),
                    "selection_pooled_auc": float(
                        selection_result[
                            "pooled_auc"
                        ]
                    ),
                    "selection_pooled_eer": float(
                        selection_result[
                            "pooled_eer"
                        ]
                    ),
                    "source_features_l2_normalized": True,
                    "fit_writers": 145,
                    "selection_writers": 36,
                    "monitor_used": False,
                    "validation_used": False,
                    "official_test_used": False,
                },
                checkpoint_path,
            )

        history_row = {
            "training_seed": int(
                training_seed
            ),
            "epoch": int(
                epoch
            ),
            "train_base_loss": float(
                epoch_summary[
                    "base_loss"
                ]
            ),
            "train_arabic_metric_loss": float(
                epoch_summary[
                    "arabic_metric_loss"
                ]
            ),
            "train_cross_metric_loss": float(
                epoch_summary[
                    "cross_metric_loss"
                ]
            ),
            "train_gradient_norm_mean": float(
                epoch_summary[
                    "gradient_norm_mean"
                ]
            ),
            "train_gradient_norm_max": float(
                epoch_summary[
                    "gradient_norm_max"
                ]
            ),
            "train_gradient_clipping_events": int(
                epoch_summary[
                    "gradient_clipping_events"
                ]
            ),
            "selection_cross_macro_auc": float(
                selection_result[
                    "cross_macro_auc"
                ]
            ),
            "selection_cross_macro_eer": float(
                selection_result[
                    "cross_macro_eer"
                ]
            ),
            "selection_pooled_auc": float(
                selection_result[
                    "pooled_auc"
                ]
            ),
            "selection_pooled_eer": float(
                selection_result[
                    "pooled_eer"
                ]
            ),
            "checkpoint_selected": bool(
                selected
            ),
        }

        for _, condition_row in (
            selection_result[
                "condition_df"
            ]
            .iterrows()
        ):
            condition = str(
                condition_row[
                    "condition"
                ]
            )

            history_row[
                f"{condition}_auc"
            ] = float(
                condition_row[
                    "auc"
                ]
            )

            history_row[
                f"{condition}_eer"
            ] = float(
                condition_row[
                    "eer"
                ]
            )

        history_rows.append(
            history_row
        )

        epoch_batch_df.to_csv(
            REPORT_DIR
            / (
                f"dinov2s_projection_seed"
                f"{training_seed}_"
                f"epoch{epoch:02d}_batches.csv"
            ),
            index=False,
        )

        print(
            f"dinov2s-projection | "
            f"seed {training_seed} | "
            f"epoch {epoch:02d} | "
            f"loss "
            f"{epoch_summary['base_loss']:.6f} | "
            f"macro AUC "
            f"{selection_result['cross_macro_auc']:.6f} | "
            f"pooled AUC "
            f"{selection_result['pooled_auc']:.6f} | "
            f"EER "
            f"{100.0 * selection_result['pooled_eer']:.4f}%"
            + (
                " | selected"
                if selected
                else ""
            )
        )

    history_df = pd.DataFrame(
        history_rows
    )

    history_path = (
        REPORT_DIR
        / (
            f"dinov2s_projection_seed"
            f"{training_seed}_history.csv"
        )
    )

    history_df.to_csv(
        history_path,
        index=False,
    )

    best_epoch_row = (
        history_df[
            history_df[
                "epoch"
            ] == best_record[
                "epoch"
            ]
        ]
        .iloc[
            0
        ]
    )

    summary = {
        "method": (
            "DINOv2-S frozen-feature projection student"
        ),
        "training_seed": int(
            training_seed
        ),
        "epochs": int(
            DINO_S_STUDENT_EPOCHS
        ),
        "projection_dimension": int(
            DINO_S_PROJECTION_DIM
        ),
        "trainable_parameters": 55440,
        "best_epoch": int(
            best_record[
                "epoch"
            ]
        ),
        "best_cross_macro_auc": float(
            best_record[
                "cross_macro_auc"
            ]
        ),
        "best_cross_macro_eer": float(
            best_record[
                "cross_macro_eer"
            ]
        ),
        "best_pooled_auc": float(
            best_record[
                "pooled_auc"
            ]
        ),
        "best_pooled_eer": float(
            best_record[
                "pooled_eer"
            ]
        ),
        "best_cross_same_same_auc": float(
            best_epoch_row[
                "cross_same_same_auc"
            ]
        ),
        "best_cross_same_variable_auc": float(
            best_epoch_row[
                "cross_same_variable_auc"
            ]
        ),
        "best_cross_variable_same_auc": float(
            best_epoch_row[
                "cross_variable_same_auc"
            ]
        ),
        "best_cross_variable_variable_auc": float(
            best_epoch_row[
                "cross_variable_variable_auc"
            ]
        ),
        "wgc_reference_cross_macro_auc": float(
            WGC_REFERENCE_CROSS_MACRO_AUC
        ),
        "pca_lda_reference_cross_macro_auc": float(
            DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
        ),
        "checkpoint_path": str(
            checkpoint_path
        ),
        "history_path": str(
            history_path
        ),
        "checkpoint_selection_events": int(
            history_df[
                "checkpoint_selected"
            ].sum()
        ),
        "selection_rule": (
            "cross_macro_auc > pooled_auc > "
            "lower_pooled_eer > earlier_epoch"
        ),
        "source_features_l2_normalized": True,
        "architecture_frozen_before_training": True,
        "optimizer_protocol_frozen_before_training": True,
        "selection_used_only_for_checkpointing": True,
        "post_hoc_hyperparameter_tuning_allowed": False,
        "monitor_used": False,
        "validation_used": False,
        "official_test_used": False,
    }

    summary_path = (
        REPORT_DIR
        / (
            f"dinov2s_projection_seed"
            f"{training_seed}_summary.json"
        )
    )

    with open(
        summary_path,
        "w",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    reproduction_model = (
        build_dinov2s_projection_student(
            training_seed
        )
        .to(
            STUDENT_DEVICE
        )
    )

    saved_checkpoint = torch.load(
        checkpoint_path,
        map_location=STUDENT_DEVICE,
        weights_only=False,
    )

    reproduction_model.load_state_dict(
        saved_checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )

    reproduction_result = (
        evaluate_projection_student_selection(
            reproduction_model
        )
    )

    reproduction_difference = {
        "cross_macro_auc": float(
            reproduction_result[
                "cross_macro_auc"
            ]
            - summary[
                "best_cross_macro_auc"
            ]
        ),
        "pooled_auc": float(
            reproduction_result[
                "pooled_auc"
            ]
            - summary[
                "best_pooled_auc"
            ]
        ),
        "pooled_eer": float(
            reproduction_result[
                "pooled_eer"
            ]
            - summary[
                "best_pooled_eer"
            ]
        ),
    }

    if (
        abs(
            reproduction_difference[
                "cross_macro_auc"
            ]
        ) > 1e-12
        or abs(
            reproduction_difference[
                "pooled_auc"
            ]
        ) > 1e-12
        or abs(
            reproduction_difference[
                "pooled_eer"
            ]
        ) > 1e-12
    ):
        raise RuntimeError(
            "Saved projection checkpoint failed exact reproduction."
        )

    summary[
        "checkpoint_reproduction_difference"
    ] = (
        reproduction_difference
    )

    with open(
        summary_path,
        "w",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    del reproduction_model
    del model
    del optimizer

    return {
        "summary": (
            summary
        ),
        "history": (
            history_df
        ),
    }


dinov2s_projection_runner_configuration = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "feature_pipeline": (
        "DINOv2-S 384-D CLS -> L2 -> "
        "Linear 384-to-144 -> L2 -> cosine"
    ),
    "training_seed": int(
        STUDENT_SEED
    ),
    "epochs": int(
        DINO_S_STUDENT_EPOCHS
    ),
    "fit_writers": 145,
    "selection_writers": 36,
    "batches_per_epoch": 18,
    "projection_dimension": int(
        DINO_S_PROJECTION_DIM
    ),
    "trainable_parameters": 55440,
    "learning_rate": float(
        DINO_S_PROJECTION_LR
    ),
    "weight_decay": float(
        DINO_S_WEIGHT_DECAY
    ),
    "cosine_margin": float(
        DINO_S_COSINE_MARGIN
    ),
    "gradient_clip_max_norm": float(
        DINO_S_GRAD_CLIP_MAX_NORM
    ),
    "objective": (
        "matched Arabic metric plus equal-mean "
        "four-condition cross-script metric"
    ),
    "checkpoint_selection_rule": (
        "cross_macro_auc > pooled_auc > "
        "lower_pooled_eer > earlier_epoch"
    ),
    "wgc_reference_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "pca_lda_reference_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "architecture_frozen": True,
    "optimizer_protocol_frozen": True,
    "batch_schedule_frozen": True,
    "selection_evaluator_frozen": bool(
        dinov2s_selection_evaluator_audit[
            "evaluator_frozen"
        ]
    ),
    "post_hoc_hyperparameter_tuning_allowed": False,
    "training_executed_in_this_cell": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_projection_runner_configuration.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_runner_configuration,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_runner_configuration,
        indent=2,
    )
)

if (
    dinov2s_projection_runner_configuration[
        "epochs"
    ] != 10
    or dinov2s_projection_runner_configuration[
        "fit_writers"
    ] != 145
    or dinov2s_projection_runner_configuration[
        "selection_writers"
    ] != 36
    or dinov2s_projection_runner_configuration[
        "batches_per_epoch"
    ] != 18
    or dinov2s_projection_runner_configuration[
        "trainable_parameters"
    ] != 55440
    or not dinov2s_projection_runner_configuration[
        "selection_evaluator_frozen"
    ]
):
    raise RuntimeError(
        "DINOv2-S projection experiment runner freeze failed."
    )

{
  "method": "DINOv2-S frozen-feature projection student",
  "feature_pipeline": "DINOv2-S 384-D CLS -> L2 -> Linear 384-to-144 -> L2 -> cosine",
  "training_seed": 42,
  "epochs": 10,
  "fit_writers": 145,
  "selection_writers": 36,
  "batches_per_epoch": 18,
  "projection_dimension": 144,
  "trainable_parameters": 55440,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "cosine_margin": 0.5,
  "gradient_clip_max_norm": 5.0,
  "objective": "matched Arabic metric plus equal-mean four-condition cross-script metric",
  "checkpoint_selection_rule": "cross_macro_auc > pooled_auc > lower_pooled_eer > earlier_epoch",
  "wgc_reference_cross_macro_auc": 0.7827656525573192,
  "pca_lda_reference_cross_macro_auc": 0.8430555555555557,
  "architecture_frozen": true,
  "optimizer_protocol_frozen": true,
  "batch_schedule_frozen": true,
  "selection_evaluator_frozen": true,
  "post_hoc_hyperparameter_tuning_allowed": false,
  "training_executed_in_this_cell": false,
  "monitor_used": false,
  "v

In [12]:
dinov2s_projection_seed42_result = (
    run_dinov2s_projection_experiment(
        training_seed=STUDENT_SEED
    )
)

dinov2s_projection_seed42_summary = (
    dinov2s_projection_seed42_result[
        "summary"
    ]
)

dinov2s_projection_seed42_history = (
    dinov2s_projection_seed42_result[
        "history"
    ]
)

print(
    "\nDINOv2-S projection seed-42 summary:"
)

print(
    json.dumps(
        dinov2s_projection_seed42_summary,
        indent=2,
    )
)

print(
    "\nEpoch history:"
)

print(
    dinov2s_projection_seed42_history[
        [
            "epoch",
            "train_base_loss",
            "train_arabic_metric_loss",
            "train_cross_metric_loss",
            "train_gradient_norm_mean",
            "selection_cross_macro_auc",
            "selection_pooled_auc",
            "selection_pooled_eer",
            "checkpoint_selected",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)

best_epoch = int(
    dinov2s_projection_seed42_summary[
        "best_epoch"
    ]
)

best_epoch_row = (
    dinov2s_projection_seed42_history[
        dinov2s_projection_seed42_history[
            "epoch"
        ] == best_epoch
    ]
    .iloc[
        0
    ]
)

best_condition_result = {
    "best_epoch": int(
        best_epoch
    ),
    "cross_same_same_auc": float(
        best_epoch_row[
            "cross_same_same_auc"
        ]
    ),
    "cross_same_variable_auc": float(
        best_epoch_row[
            "cross_same_variable_auc"
        ]
    ),
    "cross_variable_same_auc": float(
        best_epoch_row[
            "cross_variable_same_auc"
        ]
    ),
    "cross_variable_variable_auc": float(
        best_epoch_row[
            "cross_variable_variable_auc"
        ]
    ),
}

print(
    "\nBest-epoch condition AUCs:"
)

print(
    best_condition_result
)

projection_vs_references = {
    "projection_cross_macro_auc": float(
        dinov2s_projection_seed42_summary[
            "best_cross_macro_auc"
        ]
    ),
    "wgc_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "projection_minus_wgc": float(
        dinov2s_projection_seed42_summary[
            "best_cross_macro_auc"
        ]
        - WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "pca_lda_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "projection_minus_pca_lda": float(
        dinov2s_projection_seed42_summary[
            "best_cross_macro_auc"
        ]
        - DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "fraction_of_wgc_to_pca_lda_gap_recovered": float(
        (
            dinov2s_projection_seed42_summary[
                "best_cross_macro_auc"
            ]
            - WGC_REFERENCE_CROSS_MACRO_AUC
        )
        / (
            DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
            - WGC_REFERENCE_CROSS_MACRO_AUC
        )
    ),
}

print(
    "\nProjection vs references:"
)

print(
    json.dumps(
        projection_vs_references,
        indent=2,
    )
)

dinov2s-projection | seed 42 | epoch 01 | loss 0.407450 | macro AUC 0.731718 | pooled AUC 0.729632 | EER 32.8770% | selected
dinov2s-projection | seed 42 | epoch 02 | loss 0.354050 | macro AUC 0.747382 | pooled AUC 0.745702 | EER 31.9444% | selected
dinov2s-projection | seed 42 | epoch 03 | loss 0.318856 | macro AUC 0.777530 | pooled AUC 0.777710 | EER 27.7778% | selected
dinov2s-projection | seed 42 | epoch 04 | loss 0.305315 | macro AUC 0.776879 | pooled AUC 0.777344 | EER 28.4722%
dinov2s-projection | seed 42 | epoch 05 | loss 0.291459 | macro AUC 0.784678 | pooled AUC 0.785243 | EER 25.6944% | selected
dinov2s-projection | seed 42 | epoch 06 | loss 0.277768 | macro AUC 0.805120 | pooled AUC 0.805361 | EER 26.3889% | selected
dinov2s-projection | seed 42 | epoch 07 | loss 0.266890 | macro AUC 0.817295 | pooled AUC 0.817842 | EER 24.8413% | selected
dinov2s-projection | seed 42 | epoch 08 | loss 0.260981 | macro AUC 0.815856 | pooled AUC 0.816372 | EER 24.4048%
dinov2s-projection | s

In [13]:
PCA_LDA_POOLED_EER_REFERENCE = (
    0.22916666666666663
)

WGC_POOLED_EER_REFERENCE = (
    0.2998015873015873
)

projection_condition_map = {
    "cross_same_same": float(
        best_epoch_row[
            "cross_same_same_auc"
        ]
    ),
    "cross_same_variable": float(
        best_epoch_row[
            "cross_same_variable_auc"
        ]
    ),
    "cross_variable_same": float(
        best_epoch_row[
            "cross_variable_same_auc"
        ]
    ),
    "cross_variable_variable": float(
        best_epoch_row[
            "cross_variable_variable_auc"
        ]
    ),
}

pca_lda_condition_map = {
    str(
        row[
            "condition"
        ]
    ): float(
        row[
            "auc"
        ]
    )
    for _, row in (
        reference_condition_df
        .iterrows()
    )
}

wgc_condition_map = {
    "cross_same_same": 0.825617,
    "cross_same_variable": 0.777976,
    "cross_variable_same": 0.738382,
    "cross_variable_variable": 0.789087,
}

condition_comparison_rows = []

for condition in sorted(
    projection_condition_map
):
    projection_auc = (
        projection_condition_map[
            condition
        ]
    )

    wgc_auc = (
        wgc_condition_map[
            condition
        ]
    )

    pca_lda_auc = (
        pca_lda_condition_map[
            condition
        ]
    )

    condition_comparison_rows.append(
        {
            "condition": (
                condition
            ),
            "wgc_auc": float(
                wgc_auc
            ),
            "projection_auc": float(
                projection_auc
            ),
            "pca_lda_auc": float(
                pca_lda_auc
            ),
            "projection_minus_wgc": float(
                projection_auc
                - wgc_auc
            ),
            "projection_minus_pca_lda": float(
                projection_auc
                - pca_lda_auc
            ),
        }
    )

projection_condition_comparison_df = (
    pd.DataFrame(
        condition_comparison_rows
    )
)

projection_macro_auc = float(
    dinov2s_projection_seed42_summary[
        "best_cross_macro_auc"
    ]
)

projection_pooled_eer = float(
    dinov2s_projection_seed42_summary[
        "best_pooled_eer"
    ]
)

projection_gain_vs_wgc = float(
    projection_macro_auc
    - WGC_REFERENCE_CROSS_MACRO_AUC
)

projection_gap_to_pca_lda = float(
    projection_macro_auc
    - DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
)

representation_gap = float(
    DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    - WGC_REFERENCE_CROSS_MACRO_AUC
)

gap_recovered_fraction = float(
    projection_gain_vs_wgc
    / representation_gap
)

conditions_better_than_wgc = int(
    (
        projection_condition_comparison_df[
            "projection_minus_wgc"
        ]
        > 0.0
    ).sum()
)

conditions_better_than_pca_lda = int(
    (
        projection_condition_comparison_df[
            "projection_minus_pca_lda"
        ]
        > 0.0
    ).sum()
)

dinov2s_projection_seed42_verdict = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "seed": 42,
    "best_epoch": int(
        dinov2s_projection_seed42_summary[
            "best_epoch"
        ]
    ),
    "trainable_parameters": int(
        dinov2s_projection_seed42_summary[
            "trainable_parameters"
        ]
    ),
    "wgc_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "projection_cross_macro_auc": float(
        projection_macro_auc
    ),
    "pca_lda_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "projection_gain_vs_wgc": float(
        projection_gain_vs_wgc
    ),
    "projection_gap_to_pca_lda": float(
        projection_gap_to_pca_lda
    ),
    "fraction_of_wgc_to_pca_lda_gap_recovered": float(
        gap_recovered_fraction
    ),
    "wgc_pooled_eer": float(
        WGC_POOLED_EER_REFERENCE
    ),
    "projection_pooled_eer": float(
        projection_pooled_eer
    ),
    "pca_lda_pooled_eer": float(
        PCA_LDA_POOLED_EER_REFERENCE
    ),
    "projection_eer_change_vs_wgc": float(
        projection_pooled_eer
        - WGC_POOLED_EER_REFERENCE
    ),
    "projection_eer_change_vs_pca_lda": float(
        projection_pooled_eer
        - PCA_LDA_POOLED_EER_REFERENCE
    ),
    "conditions_better_than_wgc": int(
        conditions_better_than_wgc
    ),
    "conditions_better_than_pca_lda": int(
        conditions_better_than_pca_lda
    ),
    "cross_conditions": 4,
    "best_epoch_is_training_boundary": bool(
        dinov2s_projection_seed42_summary[
            "best_epoch"
        ]
        == DINO_S_STUDENT_EPOCHS
    ),
    "extend_epoch_budget_post_hoc": False,
    "seed42_result_interpretation": (
        "strong_positive_baseline"
    ),
    "run_multiseed_next": True,
    "partial_backbone_finetuning_now": False,
    "reason_for_multiseed": (
        "verify robustness to projection initialization "
        "before increasing backbone complexity"
    ),
    "architecture_changed_after_result": False,
    "learning_rate_changed_after_result": False,
    "epoch_budget_changed_after_result": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

projection_condition_comparison_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_seed42_condition_comparison.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_seed42_verdict.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_seed42_verdict,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_seed42_verdict,
        indent=2,
    )
)

print(
    "\nCondition comparison:"
)

print(
    projection_condition_comparison_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

{
  "method": "DINOv2-S frozen-feature projection student",
  "seed": 42,
  "best_epoch": 10,
  "trainable_parameters": 55440,
  "wgc_cross_macro_auc": 0.7827656525573192,
  "projection_cross_macro_auc": 0.8374834656084656,
  "pca_lda_cross_macro_auc": 0.8430555555555557,
  "projection_gain_vs_wgc": 0.054717813051146424,
  "projection_gap_to_pca_lda": -0.00557208994709002,
  "fraction_of_wgc_to_pca_lda_gap_recovered": 0.9075783892494733,
  "wgc_pooled_eer": 0.2998015873015873,
  "projection_pooled_eer": 0.24285714285714285,
  "pca_lda_pooled_eer": 0.22916666666666663,
  "projection_eer_change_vs_wgc": -0.056944444444444436,
  "projection_eer_change_vs_pca_lda": 0.013690476190476225,
  "conditions_better_than_wgc": 4,
  "conditions_better_than_pca_lda": 2,
  "cross_conditions": 4,
  "best_epoch_is_training_boundary": true,
  "extend_epoch_budget_post_hoc": false,
  "seed42_result_interpretation": "strong_positive_baseline",
  "run_multiseed_next": true,
  "partial_backbone_finetuning_no

In [14]:
MULTISEED_SEEDS = [
    42,
    123,
    2026,
]

multiseed_results = {
    42: (
        dinov2s_projection_seed42_result
    )
}

for training_seed in [
    123,
    2026,
]:
    print(
        "\n"
        + "=" * 88
    )

    print(
        f"Running frozen DINOv2-S projection student | seed {training_seed}"
    )

    print(
        "=" * 88
    )

    multiseed_results[
        training_seed
    ] = (
        run_dinov2s_projection_experiment(
            training_seed=training_seed
        )
    )


multiseed_rows = []

for training_seed in (
    MULTISEED_SEEDS
):
    summary = (
        multiseed_results[
            training_seed
        ][
            "summary"
        ]
    )

    macro_auc = float(
        summary[
            "best_cross_macro_auc"
        ]
    )

    gain_vs_wgc = float(
        macro_auc
        - WGC_REFERENCE_CROSS_MACRO_AUC
    )

    gap_to_pca_lda = float(
        macro_auc
        - DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    )

    recovered_fraction = float(
        gain_vs_wgc
        / (
            DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
            - WGC_REFERENCE_CROSS_MACRO_AUC
        )
    )

    multiseed_rows.append(
        {
            "seed": int(
                training_seed
            ),
            "best_epoch": int(
                summary[
                    "best_epoch"
                ]
            ),
            "cross_macro_auc": float(
                macro_auc
            ),
            "pooled_auc": float(
                summary[
                    "best_pooled_auc"
                ]
            ),
            "pooled_eer": float(
                summary[
                    "best_pooled_eer"
                ]
            ),
            "gain_vs_wgc": float(
                gain_vs_wgc
            ),
            "gap_to_pca_lda": float(
                gap_to_pca_lda
            ),
            "fraction_gap_recovered": float(
                recovered_fraction
            ),
            "cross_same_same_auc": float(
                summary[
                    "best_cross_same_same_auc"
                ]
            ),
            "cross_same_variable_auc": float(
                summary[
                    "best_cross_same_variable_auc"
                ]
            ),
            "cross_variable_same_auc": float(
                summary[
                    "best_cross_variable_same_auc"
                ]
            ),
            "cross_variable_variable_auc": float(
                summary[
                    "best_cross_variable_variable_auc"
                ]
            ),
        }
    )


dinov2s_projection_multiseed_df = (
    pd.DataFrame(
        multiseed_rows
    )
)

condition_columns = [
    "cross_same_same_auc",
    "cross_same_variable_auc",
    "cross_variable_same_auc",
    "cross_variable_variable_auc",
]

condition_summary_rows = []

for condition_column in (
    condition_columns
):
    condition_name = (
        condition_column
        .removesuffix(
            "_auc"
        )
    )

    values = (
        dinov2s_projection_multiseed_df[
            condition_column
        ]
        .to_numpy()
    )

    condition_summary_rows.append(
        {
            "condition": (
                condition_name
            ),
            "mean_auc": float(
                values.mean()
            ),
            "std_auc": float(
                values.std(
                    ddof=1
                )
            ),
            "min_auc": float(
                values.min()
            ),
            "max_auc": float(
                values.max()
            ),
            "wgc_auc": float(
                wgc_condition_map[
                    condition_name
                ]
            ),
            "mean_gain_vs_wgc": float(
                values.mean()
                - wgc_condition_map[
                    condition_name
                ]
            ),
            "all_seeds_above_wgc": bool(
                (
                    values
                    > wgc_condition_map[
                        condition_name
                    ]
                ).all()
            ),
        }
    )


dinov2s_projection_multiseed_condition_df = (
    pd.DataFrame(
        condition_summary_rows
    )
)

macro_values = (
    dinov2s_projection_multiseed_df[
        "cross_macro_auc"
    ]
    .to_numpy()
)

pooled_auc_values = (
    dinov2s_projection_multiseed_df[
        "pooled_auc"
    ]
    .to_numpy()
)

pooled_eer_values = (
    dinov2s_projection_multiseed_df[
        "pooled_eer"
    ]
    .to_numpy()
)

gain_values = (
    dinov2s_projection_multiseed_df[
        "gain_vs_wgc"
    ]
    .to_numpy()
)

recovered_fraction_values = (
    dinov2s_projection_multiseed_df[
        "fraction_gap_recovered"
    ]
    .to_numpy()
)


dinov2s_projection_multiseed_summary = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "seeds": (
        MULTISEED_SEEDS
    ),
    "seed_count": int(
        len(
            MULTISEED_SEEDS
        )
    ),
    "cross_macro_auc_mean": float(
        macro_values.mean()
    ),
    "cross_macro_auc_std": float(
        macro_values.std(
            ddof=1
        )
    ),
    "cross_macro_auc_min": float(
        macro_values.min()
    ),
    "cross_macro_auc_max": float(
        macro_values.max()
    ),
    "pooled_auc_mean": float(
        pooled_auc_values.mean()
    ),
    "pooled_auc_std": float(
        pooled_auc_values.std(
            ddof=1
        )
    ),
    "pooled_eer_mean": float(
        pooled_eer_values.mean()
    ),
    "pooled_eer_std": float(
        pooled_eer_values.std(
            ddof=1
        )
    ),
    "wgc_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "pca_lda_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "mean_gain_vs_wgc": float(
        gain_values.mean()
    ),
    "minimum_gain_vs_wgc": float(
        gain_values.min()
    ),
    "maximum_gain_vs_wgc": float(
        gain_values.max()
    ),
    "all_seeds_above_wgc": bool(
        (
            macro_values
            > WGC_REFERENCE_CROSS_MACRO_AUC
        ).all()
    ),
    "seeds_above_wgc": int(
        (
            macro_values
            > WGC_REFERENCE_CROSS_MACRO_AUC
        ).sum()
    ),
    "seeds_above_pca_lda": int(
        (
            macro_values
            > DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
        ).sum()
    ),
    "mean_fraction_of_wgc_to_pca_lda_gap_recovered": float(
        recovered_fraction_values.mean()
    ),
    "minimum_fraction_of_gap_recovered": float(
        recovered_fraction_values.min()
    ),
    "maximum_fraction_of_gap_recovered": float(
        recovered_fraction_values.max()
    ),
    "best_epochs": (
        dinov2s_projection_multiseed_df[
            "best_epoch"
        ]
        .astype(int)
        .tolist()
    ),
    "seeds_best_at_epoch_boundary": int(
        (
            dinov2s_projection_multiseed_df[
                "best_epoch"
            ]
            == DINO_S_STUDENT_EPOCHS
        ).sum()
    ),
    "all_conditions_all_seeds_above_wgc": bool(
        dinov2s_projection_multiseed_condition_df[
            "all_seeds_above_wgc"
        ].all()
    ),
    "architecture_identical_across_seeds": True,
    "optimizer_protocol_identical_across_seeds": True,
    "batch_schedule_identical_across_seeds": True,
    "selection_rule_identical_across_seeds": True,
    "epoch_budget_extended": False,
    "hyperparameters_changed_between_seeds": False,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}


dinov2s_projection_multiseed_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_multiseed_results.csv",
    index=False,
)

dinov2s_projection_multiseed_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_projection_multiseed_condition_summary.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_multiseed_summary.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_multiseed_summary,
        file,
        indent=2,
    )


print(
    "\nPer-seed results:"
)

print(
    dinov2s_projection_multiseed_df[
        [
            "seed",
            "best_epoch",
            "cross_macro_auc",
            "pooled_auc",
            "pooled_eer",
            "gain_vs_wgc",
            "gap_to_pca_lda",
            "fraction_gap_recovered",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)

print(
    "\n3-seed summary:"
)

print(
    json.dumps(
        dinov2s_projection_multiseed_summary,
        indent=2,
    )
)

print(
    "\nCondition robustness:"
)

print(
    dinov2s_projection_multiseed_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        dinov2s_projection_multiseed_df
    ) != 3
    or set(
        dinov2s_projection_multiseed_df[
            "seed"
        ].astype(int)
    )
    != {
        42,
        123,
        2026,
    }
    or not np.isfinite(
        dinov2s_projection_multiseed_df[
            [
                "cross_macro_auc",
                "pooled_auc",
                "pooled_eer",
            ]
        ].to_numpy()
    ).all()
):
    raise RuntimeError(
        "DINOv2-S projection multiseed audit failed."
    )


Running frozen DINOv2-S projection student | seed 123
dinov2s-projection | seed 123 | epoch 01 | loss 0.406024 | macro AUC 0.747878 | pooled AUC 0.748012 | EER 31.9444% | selected
dinov2s-projection | seed 123 | epoch 02 | loss 0.352396 | macro AUC 0.760532 | pooled AUC 0.760458 | EER 29.8611% | selected
dinov2s-projection | seed 123 | epoch 03 | loss 0.319155 | macro AUC 0.786635 | pooled AUC 0.787322 | EER 28.4524% | selected
dinov2s-projection | seed 123 | epoch 04 | loss 0.306042 | macro AUC 0.789374 | pooled AUC 0.790074 | EER 27.1032% | selected
dinov2s-projection | seed 123 | epoch 05 | loss 0.292526 | macro AUC 0.797658 | pooled AUC 0.798714 | EER 25.3770% | selected
dinov2s-projection | seed 123 | epoch 06 | loss 0.278808 | macro AUC 0.812869 | pooled AUC 0.812795 | EER 25.6944% | selected
dinov2s-projection | seed 123 | epoch 07 | loss 0.268901 | macro AUC 0.821776 | pooled AUC 0.822067 | EER 25.0000% | selected
dinov2s-projection | seed 123 | epoch 08 | loss 0.263835 | macr

In [15]:
PROJECTION_MEAN_GAIN_THRESHOLD = 0.03

PROJECTION_MINIMUM_SEED_GAIN_THRESHOLD = 0.03

PROJECTION_MAX_REFERENCE_GAP_FOR_STRONG_BASELINE = 0.01

projection_mean_macro_auc = float(
    dinov2s_projection_multiseed_summary[
        "cross_macro_auc_mean"
    ]
)

projection_macro_std = float(
    dinov2s_projection_multiseed_summary[
        "cross_macro_auc_std"
    ]
)

projection_mean_gain_vs_wgc = float(
    dinov2s_projection_multiseed_summary[
        "mean_gain_vs_wgc"
    ]
)

projection_minimum_gain_vs_wgc = float(
    dinov2s_projection_multiseed_summary[
        "minimum_gain_vs_wgc"
    ]
)

projection_mean_gap_to_reference = float(
    projection_mean_macro_auc
    - DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
)

projection_absolute_reference_gap = float(
    abs(
        projection_mean_gap_to_reference
    )
)

mean_gain_passed = bool(
    projection_mean_gain_vs_wgc
    >= PROJECTION_MEAN_GAIN_THRESHOLD
)

minimum_seed_gain_passed = bool(
    projection_minimum_gain_vs_wgc
    >= PROJECTION_MINIMUM_SEED_GAIN_THRESHOLD
)

all_seed_condition_guardrail_passed = bool(
    dinov2s_projection_multiseed_summary[
        "all_conditions_all_seeds_above_wgc"
    ]
)

reference_gap_small = bool(
    projection_absolute_reference_gap
    <= PROJECTION_MAX_REFERENCE_GAP_FOR_STRONG_BASELINE
)

multiseed_stability_supported = bool(
    dinov2s_projection_multiseed_summary[
        "all_seeds_above_wgc"
    ]
    and mean_gain_passed
    and minimum_seed_gain_passed
    and all_seed_condition_guardrail_passed
)

projection_baseline_supported = bool(
    multiseed_stability_supported
    and reference_gap_small
)

dinov2s_projection_multiseed_verdict = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "seeds": [
        42,
        123,
        2026,
    ],
    "cross_macro_auc_mean": float(
        projection_mean_macro_auc
    ),
    "cross_macro_auc_std": float(
        projection_macro_std
    ),
    "cross_macro_auc_min": float(
        dinov2s_projection_multiseed_summary[
            "cross_macro_auc_min"
        ]
    ),
    "cross_macro_auc_max": float(
        dinov2s_projection_multiseed_summary[
            "cross_macro_auc_max"
        ]
    ),
    "wgc_cross_macro_auc": float(
        WGC_REFERENCE_CROSS_MACRO_AUC
    ),
    "mean_gain_vs_wgc": float(
        projection_mean_gain_vs_wgc
    ),
    "minimum_seed_gain_vs_wgc": float(
        projection_minimum_gain_vs_wgc
    ),
    "mean_gain_threshold": float(
        PROJECTION_MEAN_GAIN_THRESHOLD
    ),
    "minimum_seed_gain_threshold": float(
        PROJECTION_MINIMUM_SEED_GAIN_THRESHOLD
    ),
    "mean_gain_passed": bool(
        mean_gain_passed
    ),
    "minimum_seed_gain_passed": bool(
        minimum_seed_gain_passed
    ),
    "all_seeds_above_wgc": bool(
        dinov2s_projection_multiseed_summary[
            "all_seeds_above_wgc"
        ]
    ),
    "all_conditions_all_seeds_above_wgc": bool(
        all_seed_condition_guardrail_passed
    ),
    "pca_lda_reference_cross_macro_auc": float(
        DINO_S_PCA_LDA_REFERENCE_CROSS_MACRO_AUC
    ),
    "mean_gap_to_pca_lda_reference": float(
        projection_mean_gap_to_reference
    ),
    "absolute_mean_reference_gap": float(
        projection_absolute_reference_gap
    ),
    "maximum_reference_gap_for_strong_baseline": float(
        PROJECTION_MAX_REFERENCE_GAP_FOR_STRONG_BASELINE
    ),
    "reference_gap_small": bool(
        reference_gap_small
    ),
    "mean_fraction_of_reference_gap_recovered": float(
        dinov2s_projection_multiseed_summary[
            "mean_fraction_of_wgc_to_pca_lda_gap_recovered"
        ]
    ),
    "pooled_eer_mean": float(
        dinov2s_projection_multiseed_summary[
            "pooled_eer_mean"
        ]
    ),
    "pooled_eer_std": float(
        dinov2s_projection_multiseed_summary[
            "pooled_eer_std"
        ]
    ),
    "best_epochs": (
        dinov2s_projection_multiseed_summary[
            "best_epochs"
        ]
    ),
    "all_seeds_best_at_epoch_boundary": bool(
        dinov2s_projection_multiseed_summary[
            "seeds_best_at_epoch_boundary"
        ]
        == 3
    ),
    "extend_training_budget_post_hoc": False,
    "multiseed_stability_supported": bool(
        multiseed_stability_supported
    ),
    "projection_baseline_supported": bool(
        projection_baseline_supported
    ),
    "partial_backbone_finetuning_now": False,
    "reason_not_to_finetune_backbone_yet": (
        "the frozen DINOv2-S representation with only "
        "55,440 trainable projection parameters already "
        "recovers most of the PCA/LDA representation advantage "
        "with low three-seed variance"
    ),
    "next_protocol_step": (
        "freeze method and perform 181-writer refit followed "
        "by one-shot evaluation on the untouched 45-writer monitor"
    ),
    "architecture_frozen": True,
    "hyperparameters_frozen": True,
    "epoch_budget_frozen": True,
    "selection_development_complete": True,
    "monitor_used": False,
    "validation_used": False,
    "official_test_used": False,
}

with open(
    REPORT_DIR
    / "dinov2s_projection_multiseed_verdict.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_multiseed_verdict,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_multiseed_verdict,
        indent=2,
    )
)

{
  "method": "DINOv2-S frozen-feature projection student",
  "seeds": [
    42,
    123,
    2026
  ],
  "cross_macro_auc_mean": 0.8374632569077014,
  "cross_macro_auc_std": 0.001854690158350641,
  "cross_macro_auc_min": 0.835598544973545,
  "cross_macro_auc_max": 0.8393077601410934,
  "wgc_cross_macro_auc": 0.7827656525573192,
  "mean_gain_vs_wgc": 0.054697604350382144,
  "minimum_seed_gain_vs_wgc": 0.052832892416225796,
  "mean_gain_threshold": 0.03,
  "minimum_seed_gain_threshold": 0.03,
  "mean_gain_passed": true,
  "minimum_seed_gain_passed": true,
  "all_seeds_above_wgc": true,
  "all_conditions_all_seeds_above_wgc": true,
  "pca_lda_reference_cross_macro_auc": 0.8430555555555557,
  "mean_gap_to_pca_lda_reference": -0.005592298647854266,
  "absolute_mean_reference_gap": 0.005592298647854266,
  "maximum_reference_gap_for_strong_baseline": 0.01,
  "reference_gap_small": true,
  "mean_fraction_of_reference_gap_recovered": 0.9072431971234408,
  "pooled_eer_mean": 0.23736772486772487

In [16]:
FINAL_REFIT_TRAINING_SEED = 42
FINAL_REFIT_SCHEDULE_SEED = 42
FINAL_REFIT_EPOCHS = 10
FINAL_REFIT_TARGET_BATCH_SIZE = 8

development_metadata = (
    split_df[
        split_df[
            "filename"
        ]
        .astype(str)
        .isin(
            set(
                dino_s_development_filenames
            )
        )
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
    .copy()
)

refit_writer_set = set(
    role_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

development_writer_set = set(
    development_metadata[
        "writer"
    ]
    .astype(int)
    .unique()
)

monitor_writer_set = (
    development_writer_set
    - refit_writer_set
)

unexpected_refit_writers = (
    refit_writer_set
    - development_writer_set
)

refit_rows = (
    development_metadata[
        development_metadata[
            "writer"
        ]
        .astype(int)
        .isin(
            refit_writer_set
        )
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
    .copy()
)

monitor_rows = (
    development_metadata[
        development_metadata[
            "writer"
        ]
        .astype(int)
        .isin(
            monitor_writer_set
        )
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
    .copy()
)

refit_page_counts = (
    refit_rows
    .groupby(
        "writer"
    )
    .size()
)

monitor_page_counts = (
    monitor_rows
    .groupby(
        "writer"
    )
    .size()
)

refit_role_audit_df = (
    development_metadata[
        [
            "filename",
            "writer",
            "page_id",
            "language",
            "same_text",
        ]
    ]
    .copy()
)

refit_role_audit_df[
    "final_role"
] = np.where(
    refit_role_audit_df[
        "writer"
    ]
    .astype(int)
    .isin(
        refit_writer_set
    ),
    "refit",
    "monitor",
)

refit_writers = np.array(
    sorted(
        refit_writer_set
    ),
    dtype=np.int64,
)

REFIT_BATCHES_PER_EPOCH = int(
    len(
        refit_writers
    )
    // FINAL_REFIT_TARGET_BATCH_SIZE
)

schedule_rng = np.random.default_rng(
    FINAL_REFIT_SCHEDULE_SEED
)

refit_schedule_rows = []

for epoch in range(
    1,
    FINAL_REFIT_EPOCHS
    + 1,
):
    epoch_writers = (
        schedule_rng.permutation(
            refit_writers
        )
    )

    epoch_batches = np.array_split(
        epoch_writers,
        REFIT_BATCHES_PER_EPOCH,
    )

    for batch_id, batch_writers in enumerate(
        epoch_batches,
        start=1,
    ):
        batch_size = int(
            len(
                batch_writers
            )
        )

        for slot, writer in enumerate(
            batch_writers,
            start=1,
        ):
            refit_schedule_rows.append(
                {
                    "epoch": int(
                        epoch
                    ),
                    "batch": int(
                        batch_id
                    ),
                    "slot": int(
                        slot
                    ),
                    "writer": int(
                        writer
                    ),
                    "batch_size": int(
                        batch_size
                    ),
                }
            )

final_refit_schedule_df = pd.DataFrame(
    refit_schedule_rows
)

rows_per_epoch = (
    final_refit_schedule_df
    .groupby(
        "epoch"
    )
    .size()
)

batches_per_epoch = (
    final_refit_schedule_df[
        [
            "epoch",
            "batch",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "epoch"
    )
    .size()
)

unique_writers_per_epoch = (
    final_refit_schedule_df
    .groupby(
        "epoch"
    )[
        "writer"
    ]
    .nunique()
)

batch_size_distribution = (
    final_refit_schedule_df[
        [
            "epoch",
            "batch",
            "batch_size",
        ]
    ]
    .drop_duplicates()[
        "batch_size"
    ]
    .value_counts()
    .sort_index()
)

dinov2s_final_refit_protocol = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "development_pages": int(
        len(
            development_metadata
        )
    ),
    "development_writers": int(
        len(
            development_writer_set
        )
    ),
    "refit_pages": int(
        len(
            refit_rows
        )
    ),
    "refit_writers": int(
        len(
            refit_writer_set
        )
    ),
    "monitor_pages": int(
        len(
            monitor_rows
        )
    ),
    "monitor_writers": int(
        len(
            monitor_writer_set
        )
    ),
    "refit_monitor_writer_overlap": int(
        len(
            refit_writer_set
            & monitor_writer_set
        )
    ),
    "unexpected_refit_writers": int(
        len(
            unexpected_refit_writers
        )
    ),
    "refit_four_pages_per_writer": bool(
        (
            refit_page_counts
            == 4
        ).all()
    ),
    "monitor_four_pages_per_writer": bool(
        (
            monitor_page_counts
            == 4
        ).all()
    ),
    "refit_language_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in (
            refit_rows[
                "language"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "monitor_language_counts": {
        str(
            key
        ): int(
            value
        )
        for key, value in (
            monitor_rows[
                "language"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "training_seed": int(
        FINAL_REFIT_TRAINING_SEED
    ),
    "schedule_seed": int(
        FINAL_REFIT_SCHEDULE_SEED
    ),
    "epochs": int(
        FINAL_REFIT_EPOCHS
    ),
    "batches_per_epoch": int(
        REFIT_BATCHES_PER_EPOCH
    ),
    "schedule_rows": int(
        len(
            final_refit_schedule_df
        )
    ),
    "all_epochs_have_181_writer_rows": bool(
        (
            rows_per_epoch
            == 181
        ).all()
    ),
    "all_epochs_have_expected_batches": bool(
        (
            batches_per_epoch
            == REFIT_BATCHES_PER_EPOCH
        ).all()
    ),
    "all_epochs_have_181_unique_writers": bool(
        (
            unique_writers_per_epoch
            == 181
        ).all()
    ),
    "batch_sizes": {
        str(
            int(
                key
            )
        ): int(
            value
        )
        for key, value in (
            batch_size_distribution
            .to_dict()
            .items()
        )
    },
    "architecture_frozen": True,
    "hyperparameters_frozen": True,
    "epoch_budget_frozen": True,
    "selection_merged_into_refit": True,
    "monitor_metadata_identified_only": True,
    "monitor_embeddings_accessed": False,
    "monitor_scores_computed": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
}

refit_role_audit_df.to_csv(
    REPORT_DIR
    / "development_final_refit_monitor_roles.csv",
    index=False,
)

final_refit_schedule_df.to_csv(
    REPORT_DIR
    / "final_refit_writer_batch_schedule_seed42.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_final_refit_protocol.json",
    "w",
) as file:
    json.dump(
        dinov2s_final_refit_protocol,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_final_refit_protocol,
        indent=2,
    )
)

print(
    "\nFinal-refit batch-size distribution:"
)

print(
    batch_size_distribution
    .to_string()
)

if (
    len(
        development_metadata
    ) != 904
    or len(
        development_writer_set
    ) != 226
    or len(
        refit_rows
    ) != 724
    or len(
        refit_writer_set
    ) != 181
    or len(
        monitor_rows
    ) != 180
    or len(
        monitor_writer_set
    ) != 45
    or len(
        refit_writer_set
        & monitor_writer_set
    ) != 0
    or len(
        unexpected_refit_writers
    ) != 0
    or not (
        refit_page_counts
        == 4
    ).all()
    or not (
        monitor_page_counts
        == 4
    ).all()
    or REFIT_BATCHES_PER_EPOCH
    != 22
    or not (
        rows_per_epoch
        == 181
    ).all()
    or not (
        batches_per_epoch
        == 22
    ).all()
    or not (
        unique_writers_per_epoch
        == 181
    ).all()
    or set(
        batch_size_distribution.index
        .astype(int)
    )
    != {
        8,
        9,
    }
):
    raise RuntimeError(
        "DINOv2-S final-refit protocol audit failed."
    )

{
  "method": "DINOv2-S frozen-feature projection student",
  "development_pages": 904,
  "development_writers": 226,
  "refit_pages": 724,
  "refit_writers": 181,
  "monitor_pages": 180,
  "monitor_writers": 45,
  "refit_monitor_writer_overlap": 0,
  "unexpected_refit_writers": 0,
  "refit_four_pages_per_writer": true,
  "monitor_four_pages_per_writer": true,
  "refit_language_counts": {
    "Arabic": 362,
    "English": 362
  },
  "monitor_language_counts": {
    "Arabic": 90,
    "English": 90
  },
  "training_seed": 42,
  "schedule_seed": 42,
  "epochs": 10,
  "batches_per_epoch": 22,
  "schedule_rows": 1810,
  "all_epochs_have_181_writer_rows": true,
  "all_epochs_have_expected_batches": true,
  "all_epochs_have_181_unique_writers": true,
  "batch_sizes": {
    "8": 170,
    "9": 50
  },
  "architecture_frozen": true,
  "hyperparameters_frozen": true,
  "epoch_budget_frozen": true,
  "selection_merged_into_refit": true,
  "monitor_metadata_identified_only": true,
  "monitor_embedd

In [17]:
fit_writer_ids = set(
    int(
        writer
    )
    for writer in (
        fit_writer_feature_map.keys()
    )
)

selection_writer_ids = set(
    selection_rows[
        "writer"
    ]
    .astype(int)
    .unique()
)

refit_writer_ids = set(
    refit_rows[
        "writer"
    ]
    .astype(int)
    .unique()
)

monitor_writer_ids = set(
    monitor_rows[
        "writer"
    ]
    .astype(int)
    .unique()
)

fit_source_rows = (
    development_metadata[
        development_metadata[
            "writer"
        ]
        .astype(int)
        .isin(
            fit_writer_ids
        )
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

fit_feature_lookup = {}

for writer, writer_rows in (
    fit_source_rows
    .groupby(
        "writer",
        sort=True,
    )
):
    writer = int(
        writer
    )

    writer_rows = (
        writer_rows
        .sort_values(
            "page_id"
        )
        .reset_index(
            drop=True
        )
    )

    writer_features = np.asarray(
        fit_writer_feature_map[
            writer
        ],
        dtype=np.float32,
    )

    page_ids = (
        writer_rows[
            "page_id"
        ]
        .astype(int)
        .tolist()
    )

    if (
        len(
            writer_rows
        ) != 4
        or writer_features.shape
        != (
            4,
            DINO_S_FEATURE_DIM,
        )
        or page_ids
        != [
            1,
            2,
            3,
            4,
        ]
    ):
        raise RuntimeError(
            f"Unexpected fit-writer feature structure for writer {writer}."
        )

    for row_index, row in (
        writer_rows
        .iterrows()
    ):
        fit_feature_lookup[
            str(
                row[
                    "filename"
                ]
            )
        ] = (
            writer_features[
                int(
                    row_index
                )
            ]
            .copy()
        )


selection_feature_lookup = {
    str(
        filename
    ): np.asarray(
        feature,
        dtype=np.float32,
    ).copy()
    for filename, feature in zip(
        selection_rows[
            "filename"
        ].astype(str),
        selection_raw_features,
    )
}


fit_filename_set = set(
    fit_feature_lookup.keys()
)

selection_filename_set = set(
    selection_feature_lookup.keys()
)

refit_filename_set = set(
    refit_rows[
        "filename"
    ]
    .astype(str)
)

monitor_filename_set = set(
    monitor_rows[
        "filename"
    ]
    .astype(str)
)


if (
    len(
        fit_writer_ids
    ) != 145
    or len(
        selection_writer_ids
    ) != 36
    or len(
        refit_writer_ids
    ) != 181
    or len(
        monitor_writer_ids
    ) != 45
    or len(
        fit_writer_ids
        & selection_writer_ids
    ) != 0
    or (
        fit_writer_ids
        | selection_writer_ids
    )
    != refit_writer_ids
    or len(
        fit_filename_set
    ) != 580
    or len(
        selection_filename_set
    ) != 144
    or len(
        fit_filename_set
        & selection_filename_set
    ) != 0
    or (
        fit_filename_set
        | selection_filename_set
    )
    != refit_filename_set
    or len(
        refit_filename_set
        & monitor_filename_set
    ) != 0
):
    raise RuntimeError(
        "Final-refit source partition audit failed."
    )


refit_feature_lookup = {}

refit_feature_lookup.update(
    fit_feature_lookup
)

refit_feature_lookup.update(
    selection_feature_lookup
)


refit_raw_features = np.stack(
    [
        refit_feature_lookup[
            filename
        ]
        for filename in (
            refit_rows[
                "filename"
            ].astype(str)
        )
    ],
    axis=0,
).astype(
    np.float32,
    copy=False,
)


refit_feature_norms = np.linalg.norm(
    refit_raw_features,
    axis=1,
)


refit_rows_with_index = (
    refit_rows
    .reset_index(
        drop=True
    )
    .copy()
)

refit_rows_with_index[
    "_refit_feature_index"
] = np.arange(
    len(
        refit_rows_with_index
    ),
    dtype=np.int64,
)


refit_writer_feature_map = {}

for writer, writer_rows in (
    refit_rows_with_index
    .groupby(
        "writer",
        sort=True,
    )
):
    writer = int(
        writer
    )

    writer_rows = (
        writer_rows
        .sort_values(
            "page_id"
        )
    )

    page_ids = (
        writer_rows[
            "page_id"
        ]
        .astype(int)
        .tolist()
    )

    feature_indices = (
        writer_rows[
            "_refit_feature_index"
        ]
        .astype(int)
        .to_numpy()
    )

    writer_features = (
        refit_raw_features[
            feature_indices
        ]
    )

    if (
        len(
            writer_rows
        ) != 4
        or page_ids
        != [
            1,
            2,
            3,
            4,
        ]
        or writer_features.shape
        != (
            4,
            DINO_S_FEATURE_DIM,
        )
    ):
        raise RuntimeError(
            f"Unexpected final-refit writer structure for writer {writer}."
        )

    refit_writer_feature_map[
        writer
    ] = (
        writer_features
        .copy()
    )


schedule_refit_writer_ids = set(
    final_refit_schedule_df[
        "writer"
    ]
    .astype(int)
    .unique()
)


def load_final_refit_writer_batch(
    epoch,
    batch_id,
):
    batch_schedule = (
        final_refit_schedule_df[
            (
                final_refit_schedule_df[
                    "epoch"
                ] == epoch
            )
            & (
                final_refit_schedule_df[
                    "batch"
                ] == batch_id
            )
        ]
        .sort_values(
            "slot"
        )
    )

    writers = (
        batch_schedule[
            "writer"
        ]
        .astype(int)
        .tolist()
    )

    features = np.stack(
        [
            refit_writer_feature_map[
                writer
            ]
            for writer in writers
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )

    return {
        "writers": writers,
        "features": features,
    }


first_refit_batch = (
    load_final_refit_writer_batch(
        epoch=1,
        batch_id=1,
    )
)


REFIT_FEATURE_ARTIFACT_PATH = (
    REPORT_DIR
    / "dinov2s_final_refit_aligned_features.npz"
)

np.savez_compressed(
    REFIT_FEATURE_ARTIFACT_PATH,
    refit_embeddings=(
        refit_raw_features
    ),
    refit_filenames=np.asarray(
        refit_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    refit_writers=np.asarray(
        refit_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    refit_page_ids=np.asarray(
        refit_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
)


dinov2s_final_refit_feature_alignment = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "source_feature_dimension": int(
        DINO_S_FEATURE_DIM
    ),
    "fit_source_writers": int(
        len(
            fit_writer_ids
        )
    ),
    "fit_source_pages": int(
        len(
            fit_feature_lookup
        )
    ),
    "selection_source_writers": int(
        len(
            selection_writer_ids
        )
    ),
    "selection_source_pages": int(
        len(
            selection_feature_lookup
        )
    ),
    "refit_writers": int(
        len(
            refit_writer_feature_map
        )
    ),
    "refit_pages": int(
        len(
            refit_raw_features
        )
    ),
    "refit_feature_shape": list(
        refit_raw_features.shape
    ),
    "refit_feature_norm_min": float(
        refit_feature_norms.min()
    ),
    "refit_feature_norm_mean": float(
        refit_feature_norms.mean()
    ),
    "refit_feature_norm_max": float(
        refit_feature_norms.max()
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_ids
            & selection_writer_ids
        )
    ),
    "refit_monitor_writer_overlap": int(
        len(
            refit_writer_ids
            & monitor_writer_ids
        )
    ),
    "refit_monitor_filename_overlap": int(
        len(
            refit_filename_set
            & monitor_filename_set
        )
    ),
    "source_writers_exactly_cover_refit": bool(
        (
            fit_writer_ids
            | selection_writer_ids
        )
        == refit_writer_ids
    ),
    "source_filenames_exactly_cover_refit": bool(
        (
            fit_filename_set
            | selection_filename_set
        )
        == refit_filename_set
    ),
    "schedule_writers_match_refit_writers": bool(
        schedule_refit_writer_ids
        == refit_writer_ids
    ),
    "all_refit_writers_have_four_features": bool(
        all(
            features.shape
            == (
                4,
                DINO_S_FEATURE_DIM,
            )
            for features in (
                refit_writer_feature_map.values()
            )
        )
    ),
    "source_features_unit_normalized": bool(
        np.allclose(
            refit_feature_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "first_batch_size": int(
        first_refit_batch[
            "features"
        ].shape[
            0
        ]
    ),
    "first_batch_feature_shape": list(
        first_refit_batch[
            "features"
        ].shape
    ),
    "aligned_feature_artifact": str(
        REFIT_FEATURE_ARTIFACT_PATH.relative_to(
            ROOT
        )
    ),
    "filename_arrays_saved_as_string_dtype": True,
    "monitor_feature_rows_used": False,
    "monitor_scores_computed": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
    "final_refit_training_started": False,
}

with open(
    REPORT_DIR
    / "dinov2s_final_refit_feature_alignment.json",
    "w",
) as file:
    json.dump(
        dinov2s_final_refit_feature_alignment,
        file,
        indent=2,
    )


print(
    json.dumps(
        dinov2s_final_refit_feature_alignment,
        indent=2,
    )
)

print(
    "\nFirst final-refit batch writers:"
)

print(
    first_refit_batch[
        "writers"
    ]
)


if (
    refit_raw_features.shape
    != (
        724,
        DINO_S_FEATURE_DIM,
    )
    or len(
        refit_writer_feature_map
    ) != 181
    or schedule_refit_writer_ids
    != refit_writer_ids
    or not np.allclose(
        refit_feature_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or len(
        refit_filename_set
        & monitor_filename_set
    ) != 0
    or not all(
        features.shape
        == (
            4,
            DINO_S_FEATURE_DIM,
        )
        for features in (
            refit_writer_feature_map.values()
        )
    )
):
    raise RuntimeError(
        "DINOv2-S final-refit feature alignment failed."
    )

{
  "method": "DINOv2-S frozen-feature projection student",
  "source_feature_dimension": 384,
  "fit_source_writers": 145,
  "fit_source_pages": 580,
  "selection_source_writers": 36,
  "selection_source_pages": 144,
  "refit_writers": 181,
  "refit_pages": 724,
  "refit_feature_shape": [
    724,
    384
  ],
  "refit_feature_norm_min": 0.9999998807907104,
  "refit_feature_norm_mean": 1.0,
  "refit_feature_norm_max": 1.0000001192092896,
  "fit_selection_writer_overlap": 0,
  "refit_monitor_writer_overlap": 0,
  "refit_monitor_filename_overlap": 0,
  "source_writers_exactly_cover_refit": true,
  "source_filenames_exactly_cover_refit": true,
  "schedule_writers_match_refit_writers": true,
  "all_refit_writers_have_four_features": true,
  "source_features_unit_normalized": true,
  "first_batch_size": 9,
  "first_batch_feature_shape": [
    9,
    4,
    384
  ],
  "aligned_feature_artifact": "reports/dinov2s_student_baseline/dinov2s_final_refit_aligned_features.npz",
  "filename_arrays_

In [18]:
def train_final_refit_batch(
    model,
    optimizer,
    epoch,
    batch_id,
):
    model.train()

    batch = (
        load_final_refit_writer_batch(
            epoch=epoch,
            batch_id=batch_id,
        )
    )

    features = (
        torch.from_numpy(
            batch[
                "features"
            ]
        )
        .float()
        .to(
            STUDENT_DEVICE
        )
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    embeddings = (
        model(
            features
        )
    )

    objective = (
        build_dinov2s_base_objective(
            embeddings
        )
    )

    loss = (
        objective[
            "base_loss"
        ]
    )

    loss.backward()

    gradient_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=(
                DINO_S_GRAD_CLIP_MAX_NORM
            ),
        )
    )

    optimizer.step()

    return {
        "epoch": int(
            epoch
        ),
        "batch": int(
            batch_id
        ),
        "batch_size": int(
            features.shape[
                0
            ]
        ),
        "base_loss": float(
            loss
            .detach()
            .cpu()
            .item()
        ),
        "arabic_metric_loss": float(
            objective[
                "arabic_result"
            ][
                "loss"
            ]
            .detach()
            .cpu()
            .item()
        ),
        "cross_metric_loss": float(
            objective[
                "cross_metric_loss"
            ]
            .detach()
            .cpu()
            .item()
        ),
        "gradient_norm_before_clip": float(
            gradient_norm
            .detach()
            .cpu()
            .item()
        ),
        "gradient_clipping_activated": bool(
            gradient_norm
            .detach()
            .cpu()
            .item()
            > DINO_S_GRAD_CLIP_MAX_NORM
        ),
    }


set_student_seed(
    FINAL_REFIT_TRAINING_SEED
)

dinov2s_final_refit_model = (
    build_dinov2s_projection_student(
        FINAL_REFIT_TRAINING_SEED
    )
    .to(
        STUDENT_DEVICE
    )
)

dinov2s_final_refit_optimizer = (
    build_dinov2s_projection_optimizer(
        dinov2s_final_refit_model
    )
)

final_refit_history_rows = []

final_refit_batch_rows = []


for epoch in range(
    1,
    FINAL_REFIT_EPOCHS
    + 1,
):
    epoch_batch_ids = (
        final_refit_schedule_df[
            final_refit_schedule_df[
                "epoch"
            ] == epoch
        ][
            "batch"
        ]
        .drop_duplicates()
        .sort_values()
        .astype(int)
        .tolist()
    )

    epoch_rows = []

    for batch_id in (
        epoch_batch_ids
    ):
        batch_result = (
            train_final_refit_batch(
                model=(
                    dinov2s_final_refit_model
                ),
                optimizer=(
                    dinov2s_final_refit_optimizer
                ),
                epoch=epoch,
                batch_id=batch_id,
            )
        )

        epoch_rows.append(
            batch_result
        )

        final_refit_batch_rows.append(
            batch_result
        )

    epoch_batch_df = (
        pd.DataFrame(
            epoch_rows
        )
    )

    epoch_summary = {
        "epoch": int(
            epoch
        ),
        "batches": int(
            len(
                epoch_batch_df
            )
        ),
        "writers_seen": int(
            epoch_batch_df[
                "batch_size"
            ].sum()
        ),
        "train_base_loss": float(
            epoch_batch_df[
                "base_loss"
            ].mean()
        ),
        "train_arabic_metric_loss": float(
            epoch_batch_df[
                "arabic_metric_loss"
            ].mean()
        ),
        "train_cross_metric_loss": float(
            epoch_batch_df[
                "cross_metric_loss"
            ].mean()
        ),
        "gradient_norm_mean": float(
            epoch_batch_df[
                "gradient_norm_before_clip"
            ].mean()
        ),
        "gradient_norm_max": float(
            epoch_batch_df[
                "gradient_norm_before_clip"
            ].max()
        ),
        "gradient_clipping_events": int(
            epoch_batch_df[
                "gradient_clipping_activated"
            ].sum()
        ),
    }

    if (
        epoch_summary[
            "batches"
        ] != 22
        or epoch_summary[
            "writers_seen"
        ] != 181
    ):
        raise RuntimeError(
            "Final-refit epoch schedule audit failed."
        )

    final_refit_history_rows.append(
        epoch_summary
    )

    print(
        f"dinov2s-final-refit | "
        f"epoch {epoch:02d} | "
        f"loss "
        f"{epoch_summary['train_base_loss']:.6f} | "
        f"arabic "
        f"{epoch_summary['train_arabic_metric_loss']:.6f} | "
        f"cross "
        f"{epoch_summary['train_cross_metric_loss']:.6f} | "
        f"grad "
        f"{epoch_summary['gradient_norm_mean']:.6f}"
    )


dinov2s_final_refit_history_df = (
    pd.DataFrame(
        final_refit_history_rows
    )
)

dinov2s_final_refit_batch_df = (
    pd.DataFrame(
        final_refit_batch_rows
    )
)


FINAL_REFIT_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "dinov2s_projection_final_refit_seed42_epoch10.pt"
)

FINAL_REFIT_HISTORY_PATH = (
    REPORT_DIR
    / "dinov2s_projection_final_refit_history.csv"
)

FINAL_REFIT_BATCH_HISTORY_PATH = (
    REPORT_DIR
    / "dinov2s_projection_final_refit_batches.csv"
)


torch.save(
    {
        "method": (
            "DINOv2-S frozen-feature projection student"
        ),
        "training_stage": (
            "final_refit"
        ),
        "training_seed": int(
            FINAL_REFIT_TRAINING_SEED
        ),
        "schedule_seed": int(
            FINAL_REFIT_SCHEDULE_SEED
        ),
        "epoch": int(
            FINAL_REFIT_EPOCHS
        ),
        "epochs_completed": int(
            FINAL_REFIT_EPOCHS
        ),
        "model_state_dict": (
            dinov2s_final_refit_model
            .state_dict()
        ),
        "optimizer_state_dict": (
            dinov2s_final_refit_optimizer
            .state_dict()
        ),
        "source_feature_dimension": int(
            DINO_S_FEATURE_DIM
        ),
        "projection_dimension": int(
            DINO_S_PROJECTION_DIM
        ),
        "trainable_parameters": 55440,
        "learning_rate": float(
            DINO_S_PROJECTION_LR
        ),
        "weight_decay": float(
            DINO_S_WEIGHT_DECAY
        ),
        "cosine_margin": float(
            DINO_S_COSINE_MARGIN
        ),
        "gradient_clip_max_norm": float(
            DINO_S_GRAD_CLIP_MAX_NORM
        ),
        "refit_writers": 181,
        "refit_pages": 724,
        "source_features_l2_normalized": True,
        "checkpoint_selection_used": False,
        "selection_evaluated_during_refit": False,
        "monitor_used": False,
        "validation_used": False,
        "official_test_used": False,
    },
    FINAL_REFIT_CHECKPOINT_PATH,
)


dinov2s_final_refit_history_df.to_csv(
    FINAL_REFIT_HISTORY_PATH,
    index=False,
)

dinov2s_final_refit_batch_df.to_csv(
    FINAL_REFIT_BATCH_HISTORY_PATH,
    index=False,
)


final_epoch_row = (
    dinov2s_final_refit_history_df
    .iloc[
        -1
    ]
)

dinov2s_final_refit_training_summary = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "training_stage": (
        "final_refit"
    ),
    "training_seed": int(
        FINAL_REFIT_TRAINING_SEED
    ),
    "schedule_seed": int(
        FINAL_REFIT_SCHEDULE_SEED
    ),
    "refit_writers": 181,
    "refit_pages": 724,
    "epochs_completed": int(
        len(
            dinov2s_final_refit_history_df
        )
    ),
    "batches_per_epoch": 22,
    "total_optimizer_steps": int(
        len(
            dinov2s_final_refit_batch_df
        )
    ),
    "initial_epoch_base_loss": float(
        dinov2s_final_refit_history_df
        .iloc[
            0
        ][
            "train_base_loss"
        ]
    ),
    "final_epoch_base_loss": float(
        final_epoch_row[
            "train_base_loss"
        ]
    ),
    "base_loss_change_epoch1_to_epoch10": float(
        final_epoch_row[
            "train_base_loss"
        ]
        - dinov2s_final_refit_history_df
        .iloc[
            0
        ][
            "train_base_loss"
        ]
    ),
    "final_epoch_arabic_metric_loss": float(
        final_epoch_row[
            "train_arabic_metric_loss"
        ]
    ),
    "final_epoch_cross_metric_loss": float(
        final_epoch_row[
            "train_cross_metric_loss"
        ]
    ),
    "gradient_norm_mean_all_batches": float(
        dinov2s_final_refit_batch_df[
            "gradient_norm_before_clip"
        ].mean()
    ),
    "gradient_norm_max_all_batches": float(
        dinov2s_final_refit_batch_df[
            "gradient_norm_before_clip"
        ].max()
    ),
    "gradient_clipping_events_total": int(
        dinov2s_final_refit_batch_df[
            "gradient_clipping_activated"
        ].sum()
    ),
    "checkpoint_path": str(
        FINAL_REFIT_CHECKPOINT_PATH
    ),
    "history_path": str(
        FINAL_REFIT_HISTORY_PATH
    ),
    "batch_history_path": str(
        FINAL_REFIT_BATCH_HISTORY_PATH
    ),
    "fixed_epoch_budget_used": True,
    "checkpoint_selection_used": False,
    "early_stopping_used": False,
    "selection_evaluated_during_refit": False,
    "selection_used_for_checkpointing_during_refit": False,
    "monitor_features_used": False,
    "monitor_scores_computed": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
    "final_refit_completed": True,
}


with open(
    REPORT_DIR
    / "dinov2s_final_refit_training_summary.json",
    "w",
) as file:
    json.dump(
        dinov2s_final_refit_training_summary,
        file,
        indent=2,
    )


print(
    "\nFinal-refit training summary:"
)

print(
    json.dumps(
        dinov2s_final_refit_training_summary,
        indent=2,
    )
)

print(
    "\nFinal-refit epoch history:"
)

print(
    dinov2s_final_refit_history_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        dinov2s_final_refit_history_df
    ) != 10
    or len(
        dinov2s_final_refit_batch_df
    ) != 220
    or not (
        dinov2s_final_refit_history_df[
            "batches"
        ]
        == 22
    ).all()
    or not (
        dinov2s_final_refit_history_df[
            "writers_seen"
        ]
        == 181
    ).all()
    or not np.isfinite(
        dinov2s_final_refit_history_df[
            [
                "train_base_loss",
                "train_arabic_metric_loss",
                "train_cross_metric_loss",
                "gradient_norm_mean",
                "gradient_norm_max",
            ]
        ].to_numpy()
    ).all()
    or not (
        FINAL_REFIT_CHECKPOINT_PATH
        .exists()
    )
):
    raise RuntimeError(
        "DINOv2-S final refit training audit failed."
    )

dinov2s-final-refit | epoch 01 | loss 0.404227 | arabic 0.188197 | cross 0.216029 | grad 0.281306
dinov2s-final-refit | epoch 02 | loss 0.348913 | arabic 0.156177 | cross 0.192736 | grad 0.341680
dinov2s-final-refit | epoch 03 | loss 0.330002 | arabic 0.145166 | cross 0.184836 | grad 0.323151
dinov2s-final-refit | epoch 04 | loss 0.307784 | arabic 0.133958 | cross 0.173826 | grad 0.384634
dinov2s-final-refit | epoch 05 | loss 0.292164 | arabic 0.123964 | cross 0.168200 | grad 0.295464
dinov2s-final-refit | epoch 06 | loss 0.284653 | arabic 0.121267 | cross 0.163386 | grad 0.289897
dinov2s-final-refit | epoch 07 | loss 0.274998 | arabic 0.116962 | cross 0.158036 | grad 0.277589
dinov2s-final-refit | epoch 08 | loss 0.271462 | arabic 0.115573 | cross 0.155889 | grad 0.304316
dinov2s-final-refit | epoch 09 | loss 0.264230 | arabic 0.111115 | cross 0.153115 | grad 0.311917
dinov2s-final-refit | epoch 10 | loss 0.258698 | arabic 0.108250 | cross 0.150448 | grad 0.351502

Final-refit trainin

In [19]:
MONITOR_SOURCE_ARCHIVE_PATH = (
    ROOT
    / "reports"
    / "dinov2_efficiency_frontier"
    / "dinov2_vits14_reg_embeddings.npz"
)

monitor_source_archive = np.load(
    MONITOR_SOURCE_ARCHIVE_PATH,
    allow_pickle=True,
)

development_embedding_candidates = []

development_filename_candidates = []

for key in (
    monitor_source_archive.files
):
    array = (
        monitor_source_archive[
            key
        ]
    )

    if (
        "embedding"
        in key.lower()
        and array.ndim == 2
        and array.shape
        == (
            904,
            DINO_S_FEATURE_DIM,
        )
        and np.issubdtype(
            array.dtype,
            np.number,
        )
    ):
        development_embedding_candidates.append(
            key
        )

    if (
        "filename"
        in key.lower()
        and array.ndim == 1
        and len(
            array
        ) == 904
    ):
        development_filename_candidates.append(
            key
        )


if (
    len(
        development_embedding_candidates
    ) != 1
    or len(
        development_filename_candidates
    ) != 1
):
    raise RuntimeError(
        "Could not uniquely identify the DINOv2-S development arrays."
    )


development_embedding_key = (
    development_embedding_candidates[
        0
    ]
)

development_filename_key = (
    development_filename_candidates[
        0
    ]
)


monitor_source_development_embeddings = (
    monitor_source_archive[
        development_embedding_key
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

monitor_source_development_filenames = (
    monitor_source_archive[
        development_filename_key
    ]
    .astype(str)
)


development_archive_filename_set = set(
    monitor_source_development_filenames
)

expected_development_filename_set = set(
    development_metadata[
        "filename"
    ]
    .astype(str)
)


if (
    monitor_source_development_embeddings.shape
    != (
        904,
        DINO_S_FEATURE_DIM,
    )
    or len(
        monitor_source_development_filenames
    ) != 904
    or development_archive_filename_set
    != expected_development_filename_set
):
    raise RuntimeError(
        "Development DINOv2-S archive alignment failed."
    )


development_source_feature_lookup = {
    filename: embedding
    for filename, embedding in zip(
        monitor_source_development_filenames,
        monitor_source_development_embeddings,
    )
}


monitor_filenames = (
    monitor_rows[
        "filename"
    ]
    .astype(str)
    .tolist()
)

missing_monitor_filenames = [
    filename
    for filename in (
        monitor_filenames
    )
    if filename
    not in development_source_feature_lookup
]


if len(
    missing_monitor_filenames
) != 0:
    raise RuntimeError(
        "Monitor filenames are missing from the DINOv2-S archive."
    )


monitor_raw_features = np.stack(
    [
        development_source_feature_lookup[
            filename
        ]
        for filename in (
            monitor_filenames
        )
    ],
    axis=0,
).astype(
    np.float32,
    copy=False,
)


monitor_raw_feature_norms = np.linalg.norm(
    monitor_raw_features,
    axis=1,
)


if (
    monitor_raw_features.shape
    != (
        180,
        DINO_S_FEATURE_DIM,
    )
    or not np.allclose(
        monitor_raw_feature_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
):
    raise RuntimeError(
        "Monitor DINOv2-S feature audit failed."
    )


monitor_model = (
    build_dinov2s_projection_student(
        FINAL_REFIT_TRAINING_SEED
    )
    .to(
        STUDENT_DEVICE
    )
)


monitor_checkpoint = torch.load(
    FINAL_REFIT_CHECKPOINT_PATH,
    map_location=STUDENT_DEVICE,
    weights_only=False,
)


monitor_model.load_state_dict(
    monitor_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)


monitor_model.eval()


with torch.no_grad():
    monitor_projected_embeddings = (
        monitor_model(
            torch.from_numpy(
                monitor_raw_features
            )
            .float()
            .to(
                STUDENT_DEVICE
            )
        )
        .detach()
        .cpu()
        .numpy()
    )


monitor_projected_norms = np.linalg.norm(
    monitor_projected_embeddings,
    axis=1,
)


monitor_embedding_lookup = {
    filename: embedding
    for filename, embedding in zip(
        monitor_filenames,
        monitor_projected_embeddings,
    )
}


monitor_page_lookup = {}

for _, row in (
    monitor_rows
    .iterrows()
):
    key = (
        int(
            row[
                "writer"
            ]
        ),
        int(
            row[
                "page_id"
            ]
        ),
    )

    if key in (
        monitor_page_lookup
    ):
        raise RuntimeError(
            "Duplicate monitor writer-page entry."
        )

    monitor_page_lookup[
        key
    ] = str(
        row[
            "filename"
        ]
    )


monitor_writer_list = sorted(
    monitor_rows[
        "writer"
    ]
    .astype(int)
    .unique()
    .tolist()
)


MONITOR_CROSS_CONDITIONS = {
    "cross_variable_variable": (
        1,
        3,
    ),
    "cross_variable_same": (
        1,
        4,
    ),
    "cross_same_variable": (
        2,
        3,
    ),
    "cross_same_same": (
        2,
        4,
    ),
}


monitor_pair_rows = []


for condition, (
    left_page_id,
    right_page_id,
) in (
    MONITOR_CROSS_CONDITIONS
    .items()
):
    for left_writer in (
        monitor_writer_list
    ):
        left_filename = (
            monitor_page_lookup[
                (
                    left_writer,
                    left_page_id,
                )
            ]
        )

        for right_writer in (
            monitor_writer_list
        ):
            right_filename = (
                monitor_page_lookup[
                    (
                        right_writer,
                        right_page_id,
                    )
                ]
            )

            score = float(
                np.dot(
                    monitor_embedding_lookup[
                        left_filename
                    ],
                    monitor_embedding_lookup[
                        right_filename
                    ],
                )
            )

            monitor_pair_rows.append(
                {
                    "condition": (
                        condition
                    ),
                    "left_writer": int(
                        left_writer
                    ),
                    "right_writer": int(
                        right_writer
                    ),
                    "left_page_id": int(
                        left_page_id
                    ),
                    "right_page_id": int(
                        right_page_id
                    ),
                    "left_filename": (
                        left_filename
                    ),
                    "right_filename": (
                        right_filename
                    ),
                    "label": int(
                        left_writer
                        == right_writer
                    ),
                    "score": float(
                        score
                    ),
                }
            )


monitor_scored_pairs_df = (
    pd.DataFrame(
        monitor_pair_rows
    )
)


monitor_condition_rows = []


for condition in (
    MONITOR_CROSS_CONDITIONS
):
    condition_df = (
        monitor_scored_pairs_df[
            monitor_scored_pairs_df[
                "condition"
            ] == condition
        ]
    )

    labels = (
        condition_df[
            "label"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    scores = (
        condition_df[
            "score"
        ]
        .to_numpy()
    )

    condition_auc = float(
        roc_auc_score(
            labels,
            scores,
        )
    )

    condition_eer, condition_threshold = (
        calculate_interpolated_eer(
            labels,
            scores,
        )
    )

    monitor_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                condition_auc
            ),
            "eer": float(
                condition_eer
            ),
            "eer_threshold": float(
                condition_threshold
            ),
            "pairs": int(
                len(
                    condition_df
                )
            ),
            "genuine_pairs": int(
                condition_df[
                    "label"
                ].sum()
            ),
            "impostor_pairs": int(
                (
                    condition_df[
                        "label"
                    ] == 0
                ).sum()
            ),
        }
    )


dinov2s_monitor_condition_df = (
    pd.DataFrame(
        monitor_condition_rows
    )
)


monitor_pooled_labels = (
    monitor_scored_pairs_df[
        "label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

monitor_pooled_scores = (
    monitor_scored_pairs_df[
        "score"
    ]
    .to_numpy()
)


monitor_pooled_auc = float(
    roc_auc_score(
        monitor_pooled_labels,
        monitor_pooled_scores,
    )
)


monitor_pooled_eer, monitor_pooled_eer_threshold = (
    calculate_interpolated_eer(
        monitor_pooled_labels,
        monitor_pooled_scores,
    )
)


monitor_cross_macro_auc = float(
    dinov2s_monitor_condition_df[
        "auc"
    ].mean()
)

monitor_cross_macro_eer = float(
    dinov2s_monitor_condition_df[
        "eer"
    ].mean()
)


selection_multiseed_macro_reference = float(
    dinov2s_projection_multiseed_summary[
        "cross_macro_auc_mean"
    ]
)


monitor_generalization_delta = float(
    monitor_cross_macro_auc
    - selection_multiseed_macro_reference
)


MONITOR_EMBEDDING_ARTIFACT_PATH = (
    REPORT_DIR
    / "dinov2s_projection_final_refit_monitor_embeddings.npz"
)

MONITOR_PAIR_PATH = (
    REPORT_DIR
    / "dinov2s_projection_final_refit_monitor_scored_pairs.csv"
)

MONITOR_CONDITION_PATH = (
    REPORT_DIR
    / "dinov2s_projection_final_refit_monitor_conditions.csv"
)


np.savez_compressed(
    MONITOR_EMBEDDING_ARTIFACT_PATH,
    monitor_embeddings=(
        monitor_projected_embeddings
    ),
    monitor_filenames=np.asarray(
        monitor_filenames,
        dtype=str,
    ),
    monitor_writers=np.asarray(
        monitor_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    monitor_page_ids=np.asarray(
        monitor_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
)


monitor_scored_pairs_df.to_csv(
    MONITOR_PAIR_PATH,
    index=False,
)

dinov2s_monitor_condition_df.to_csv(
    MONITOR_CONDITION_PATH,
    index=False,
)


dinov2s_final_refit_monitor_summary = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "training_stage": (
        "181-writer final refit"
    ),
    "evaluation_stage": (
        "one-shot 45-writer monitor"
    ),
    "training_seed": int(
        FINAL_REFIT_TRAINING_SEED
    ),
    "checkpoint_epoch": int(
        monitor_checkpoint[
            "epoch"
        ]
    ),
    "monitor_writers": int(
        len(
            monitor_writer_list
        )
    ),
    "monitor_pages": int(
        len(
            monitor_rows
        )
    ),
    "source_feature_shape": list(
        monitor_raw_features.shape
    ),
    "projected_embedding_shape": list(
        monitor_projected_embeddings.shape
    ),
    "source_feature_norm_min": float(
        monitor_raw_feature_norms.min()
    ),
    "source_feature_norm_mean": float(
        monitor_raw_feature_norms.mean()
    ),
    "source_feature_norm_max": float(
        monitor_raw_feature_norms.max()
    ),
    "projected_norm_min": float(
        monitor_projected_norms.min()
    ),
    "projected_norm_mean": float(
        monitor_projected_norms.mean()
    ),
    "projected_norm_max": float(
        monitor_projected_norms.max()
    ),
    "cross_conditions": int(
        len(
            dinov2s_monitor_condition_df
        )
    ),
    "total_cross_pairs": int(
        len(
            monitor_scored_pairs_df
        )
    ),
    "pairs_per_condition": int(
        len(
            monitor_writer_list
        )
        ** 2
    ),
    "genuine_pairs_per_condition": int(
        len(
            monitor_writer_list
        )
    ),
    "impostor_pairs_per_condition": int(
        len(
            monitor_writer_list
        )
        * (
            len(
                monitor_writer_list
            )
            - 1
        )
    ),
    "monitor_cross_macro_auc": float(
        monitor_cross_macro_auc
    ),
    "monitor_cross_macro_eer": float(
        monitor_cross_macro_eer
    ),
    "monitor_pooled_auc": float(
        monitor_pooled_auc
    ),
    "monitor_pooled_eer": float(
        monitor_pooled_eer
    ),
    "monitor_pooled_eer_threshold": float(
        monitor_pooled_eer_threshold
    ),
    "selection_multiseed_macro_auc_reference": float(
        selection_multiseed_macro_reference
    ),
    "monitor_minus_selection_multiseed_macro_auc": float(
        monitor_generalization_delta
    ),
    "model_frozen_before_monitor": True,
    "architecture_frozen_before_monitor": True,
    "hyperparameters_frozen_before_monitor": True,
    "epoch_budget_frozen_before_monitor": True,
    "monitor_used_for_model_selection": False,
    "monitor_used_for_checkpoint_selection": False,
    "monitor_one_shot_evaluation": True,
    "post_monitor_hyperparameter_tuning_allowed": False,
    "validation_used": False,
    "official_test_used": False,
}


with open(
    REPORT_DIR
    / "dinov2s_final_refit_monitor_summary.json",
    "w",
) as file:
    json.dump(
        dinov2s_final_refit_monitor_summary,
        file,
        indent=2,
    )


print(
    json.dumps(
        dinov2s_final_refit_monitor_summary,
        indent=2,
    )
)

print(
    "\nMonitor condition results:"
)

print(
    dinov2s_monitor_condition_df[
        [
            "condition",
            "auc",
            "eer",
            "pairs",
            "genuine_pairs",
            "impostor_pairs",
        ]
    ]
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    len(
        monitor_writer_list
    ) != 45
    or len(
        monitor_rows
    ) != 180
    or monitor_raw_features.shape
    != (
        180,
        DINO_S_FEATURE_DIM,
    )
    or monitor_projected_embeddings.shape
    != (
        180,
        DINO_S_PROJECTION_DIM,
    )
    or len(
        monitor_scored_pairs_df
    ) != 8100
    or not (
        dinov2s_monitor_condition_df[
            "pairs"
        ]
        == 2025
    ).all()
    or not (
        dinov2s_monitor_condition_df[
            "genuine_pairs"
        ]
        == 45
    ).all()
    or not (
        dinov2s_monitor_condition_df[
            "impostor_pairs"
        ]
        == 1980
    ).all()
    or not np.allclose(
        monitor_projected_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
    or not np.isfinite(
        [
            monitor_cross_macro_auc,
            monitor_cross_macro_eer,
            monitor_pooled_auc,
            monitor_pooled_eer,
        ]
    ).all()
):
    raise RuntimeError(
        "DINOv2-S one-shot monitor evaluation audit failed."
    )

{
  "method": "DINOv2-S frozen-feature projection student",
  "training_stage": "181-writer final refit",
  "evaluation_stage": "one-shot 45-writer monitor",
  "training_seed": 42,
  "checkpoint_epoch": 10,
  "monitor_writers": 45,
  "monitor_pages": 180,
  "source_feature_shape": [
    180,
    384
  ],
  "projected_embedding_shape": [
    180,
    144
  ],
  "source_feature_norm_min": 0.9999998807907104,
  "source_feature_norm_mean": 1.0,
  "source_feature_norm_max": 1.0000001192092896,
  "projected_norm_min": 0.9999998807907104,
  "projected_norm_mean": 1.0,
  "projected_norm_max": 1.0000001192092896,
  "cross_conditions": 4,
  "total_cross_pairs": 8100,
  "pairs_per_condition": 2025,
  "genuine_pairs_per_condition": 45,
  "impostor_pairs_per_condition": 1980,
  "monitor_cross_macro_auc": 0.8538103254769922,
  "monitor_cross_macro_eer": 0.22777777777777777,
  "monitor_pooled_auc": 0.8540523288439954,
  "monitor_pooled_eer": 0.2244949494949495,
  "monitor_pooled_eer_threshold": 0.685

In [20]:
HISTORICAL_WGC_MONITOR_MACRO_AUC = (
    0.7517649
)

monitor_condition_auc_map = {
    str(
        row[
            "condition"
        ]
    ): float(
        row[
            "auc"
        ]
    )
    for _, row in (
        dinov2s_monitor_condition_df
        .iterrows()
    )
}

selection_multiseed_condition_mean_map = {
    str(
        row[
            "condition"
        ]
    ): float(
        row[
            "mean_auc"
        ]
    )
    for _, row in (
        dinov2s_projection_multiseed_condition_df
        .iterrows()
    )
}

monitor_condition_comparison_rows = []

for condition in sorted(
    monitor_condition_auc_map
):
    monitor_auc = (
        monitor_condition_auc_map[
            condition
        ]
    )

    selection_mean_auc = (
        selection_multiseed_condition_mean_map[
            condition
        ]
    )

    monitor_condition_comparison_rows.append(
        {
            "condition": (
                condition
            ),
            "selection_multiseed_mean_auc": float(
                selection_mean_auc
            ),
            "monitor_auc": float(
                monitor_auc
            ),
            "monitor_minus_selection_mean": float(
                monitor_auc
                - selection_mean_auc
            ),
        }
    )

dinov2s_monitor_vs_selection_condition_df = (
    pd.DataFrame(
        monitor_condition_comparison_rows
    )
)

selection_macro_mean = float(
    dinov2s_projection_multiseed_summary[
        "cross_macro_auc_mean"
    ]
)

selection_macro_std = float(
    dinov2s_projection_multiseed_summary[
        "cross_macro_auc_std"
    ]
)

monitor_macro_auc = float(
    dinov2s_final_refit_monitor_summary[
        "monitor_cross_macro_auc"
    ]
)

monitor_macro_eer = float(
    dinov2s_final_refit_monitor_summary[
        "monitor_cross_macro_eer"
    ]
)

monitor_pooled_auc = float(
    dinov2s_final_refit_monitor_summary[
        "monitor_pooled_auc"
    ]
)

monitor_pooled_eer = float(
    dinov2s_final_refit_monitor_summary[
        "monitor_pooled_eer"
    ]
)

monitor_minus_selection = float(
    monitor_macro_auc
    - selection_macro_mean
)

monitor_gain_vs_historical_wgc = float(
    monitor_macro_auc
    - HISTORICAL_WGC_MONITOR_MACRO_AUC
)

selection_robustness_passed = bool(
    dinov2s_projection_multiseed_verdict[
        "projection_baseline_supported"
    ]
)

monitor_generalization_supported = bool(
    monitor_macro_auc
    >= selection_macro_mean
    - 0.01
)

representation_pivot_supported = bool(
    selection_robustness_passed
    and monitor_generalization_supported
    and monitor_gain_vs_historical_wgc
    > 0.05
)

dinov2s_projection_final_verdict = {
    "method": (
        "DINOv2-S frozen-feature projection student"
    ),
    "architecture": (
        "cached normalized DINOv2-S 384-D features "
        "-> trainable Linear 384-to-144 "
        "-> L2 normalization -> cosine verification"
    ),
    "trainable_projection_parameters": 55440,
    "selection_seeds": [
        42,
        123,
        2026,
    ],
    "selection_cross_macro_auc_mean": float(
        selection_macro_mean
    ),
    "selection_cross_macro_auc_std": float(
        selection_macro_std
    ),
    "selection_cross_macro_auc_min": float(
        dinov2s_projection_multiseed_summary[
            "cross_macro_auc_min"
        ]
    ),
    "selection_cross_macro_auc_max": float(
        dinov2s_projection_multiseed_summary[
            "cross_macro_auc_max"
        ]
    ),
    "selection_all_seeds_above_wgc": bool(
        dinov2s_projection_multiseed_summary[
            "all_seeds_above_wgc"
        ]
    ),
    "selection_all_conditions_all_seeds_above_wgc": bool(
        dinov2s_projection_multiseed_summary[
            "all_conditions_all_seeds_above_wgc"
        ]
    ),
    "selection_mean_fraction_of_pca_lda_gap_recovered": float(
        dinov2s_projection_multiseed_summary[
            "mean_fraction_of_wgc_to_pca_lda_gap_recovered"
        ]
    ),
    "final_refit_writers": 181,
    "final_refit_pages": 724,
    "monitor_writers": 45,
    "monitor_pages": 180,
    "monitor_cross_macro_auc": float(
        monitor_macro_auc
    ),
    "monitor_cross_macro_eer": float(
        monitor_macro_eer
    ),
    "monitor_pooled_auc": float(
        monitor_pooled_auc
    ),
    "monitor_pooled_eer": float(
        monitor_pooled_eer
    ),
    "monitor_minus_selection_multiseed_macro_auc": float(
        monitor_minus_selection
    ),
    "historical_wgc_monitor_macro_auc": float(
        HISTORICAL_WGC_MONITOR_MACRO_AUC
    ),
    "monitor_gain_vs_historical_wgc": float(
        monitor_gain_vs_historical_wgc
    ),
    "monitor_conditions_above_0_80_auc": int(
        (
            dinov2s_monitor_condition_df[
                "auc"
            ]
            > 0.80
        ).sum()
    ),
    "monitor_cross_conditions": 4,
    "selection_robustness_supported": bool(
        selection_robustness_passed
    ),
    "monitor_generalization_supported": bool(
        monitor_generalization_supported
    ),
    "representation_pivot_supported": bool(
        representation_pivot_supported
    ),
    "result_interpretation": (
        "strong positive evidence that the earlier "
        "ResNet-based student was representation-limited; "
        "a frozen DINOv2-S representation with a very small "
        "trainable projection head generalizes strongly "
        "across scripts and writers"
    ),
    "partial_backbone_finetuning_required_for_this_baseline": False,
    "extend_epoch_budget_post_hoc": False,
    "post_monitor_architecture_tuning_allowed": False,
    "post_monitor_hyperparameter_tuning_allowed": False,
    "selection_phase_closed": True,
    "monitor_phase_closed": True,
    "validation_used": False,
    "official_test_used": False,
}

dinov2s_monitor_vs_selection_condition_df.to_csv(
    REPORT_DIR
    / "dinov2s_monitor_vs_selection_condition_comparison.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_projection_final_verdict.json",
    "w",
) as file:
    json.dump(
        dinov2s_projection_final_verdict,
        file,
        indent=2,
    )

print(
    json.dumps(
        dinov2s_projection_final_verdict,
        indent=2,
    )
)

print(
    "\nMonitor vs selection-condition means:"
)

print(
    dinov2s_monitor_vs_selection_condition_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)

if (
    not dinov2s_projection_final_verdict[
        "selection_robustness_supported"
    ]
    or not dinov2s_projection_final_verdict[
        "monitor_generalization_supported"
    ]
    or dinov2s_projection_final_verdict[
        "monitor_writers"
    ] != 45
    or dinov2s_projection_final_verdict[
        "monitor_cross_conditions"
    ] != 4
):
    raise RuntimeError(
        "DINOv2-S projection final verdict audit failed."
    )

{
  "method": "DINOv2-S frozen-feature projection student",
  "architecture": "cached normalized DINOv2-S 384-D features -> trainable Linear 384-to-144 -> L2 normalization -> cosine verification",
  "trainable_projection_parameters": 55440,
  "selection_seeds": [
    42,
    123,
    2026
  ],
  "selection_cross_macro_auc_mean": 0.8374632569077014,
  "selection_cross_macro_auc_std": 0.001854690158350641,
  "selection_cross_macro_auc_min": 0.835598544973545,
  "selection_cross_macro_auc_max": 0.8393077601410934,
  "selection_all_seeds_above_wgc": true,
  "selection_all_conditions_all_seeds_above_wgc": true,
  "selection_mean_fraction_of_pca_lda_gap_recovered": 0.9072431971234408,
  "final_refit_writers": 181,
  "final_refit_pages": 724,
  "monitor_writers": 45,
  "monitor_pages": 180,
  "monitor_cross_macro_auc": 0.8538103254769922,
  "monitor_cross_macro_eer": 0.22777777777777777,
  "monitor_pooled_auc": 0.8540523288439954,
  "monitor_pooled_eer": 0.2244949494949495,
  "monitor_minus_s

## Final Summary

This notebook tested whether the previously observed cross-script performance ceiling was primarily caused by the student representation rather than by the metric-learning objective.

A lightweight student was constructed using the cached DINOv2-S representation:

**normalized DINOv2-S 384-D feature → trainable Linear 384→144 projection → L2 normalization → cosine verification**

Only **55,440 parameters** were trainable. The DINOv2-S representation itself remained frozen.

The projection architecture, optimizer, learning rate, margin, 10-epoch budget, writer-centric batch schedule, and checkpoint-selection rule were fixed before the main experiment.

### Development Selection Result

Using the 145-writer fit split and the 36-writer selection split, the frozen protocol was evaluated with three predetermined seeds: **42, 123, and 2026**.

The resulting cross-script macro AUC was:

**0.83746 ± 0.00185**

with a range of:

**0.83560–0.83931**

All three seeds outperformed the previous ResNet18 WGC reference (**0.78277**), and every cross-script condition remained above the corresponding WGC result for every seed.

The mean gain over WGC was:

**+0.05470 macro AUC**

The simple projection recovered approximately:

**90.72%**

of the performance gap between the WGC baseline and the supervised DINOv2-S PCA/LDA reference (**0.84306 macro AUC**).

This indicates that most of the previously missing verification performance was already present in the DINOv2-S representation and could be exposed using only a very small trainable metric head.

### Clean Final Refit and Monitor Evaluation

After the method was frozen, the original 145 fit writers and 36 selection writers were merged into a **181-writer final-refit set**.

The projection student was retrained from initialization for the fixed 10-epoch budget using all 181 writers.

No selection-based checkpointing, early stopping, epoch extension, or hyperparameter adjustment was performed during this final refit.

The resulting epoch-10 model was then evaluated once on the untouched **45-writer monitor set**.

Monitor performance was:

- Cross-script macro AUC: **0.85381**
- Cross-script macro EER: **0.22778**
- Pooled AUC: **0.85405**
- Pooled EER: **0.22449**

The monitor macro AUC was **+0.01635** above the three-seed selection mean, providing no evidence of a selection-to-monitor generalization collapse.

The four monitor condition AUCs were:

| Condition | AUC | EER |
|---|---:|---:|
| cross_same_same | 0.86959 | 0.22222 |
| cross_same_variable | 0.86548 | 0.22222 |
| cross_variable_same | 0.83449 | 0.24444 |
| cross_variable_variable | 0.84569 | 0.22222 |

All four monitor conditions achieved AUC above **0.80**.

Compared with the historical WGC monitor macro AUC of **0.75176**, the DINOv2-S projection student improved macro AUC by approximately:

**+0.10205**

or about **10.2 AUC percentage points**.

### Scientific Interpretation

The results strongly support the representation-bottleneck hypothesis.

Earlier attempts to improve the ResNet18 student through geometry consistency and teacher-guided objectives produced comparatively limited gains. In contrast, replacing the underlying representation with frozen DINOv2-S features produced a large and highly stable improvement while requiring only a small linear projection head.

Therefore, the evidence suggests that the earlier student was primarily limited by the quality of its representation rather than by the absence of a more complex distillation or invariance objective.

The result also shows that full DINOv2-S backbone fine-tuning is not required to establish a strong cross-script student baseline. A frozen DINOv2-S representation plus a small trainable projection is already sufficient to obtain strong writer-disjoint generalization.

This model should still be described precisely as a **frozen-feature projection student**, not as an end-to-end fine-tuned DINOv2-S model.

### Protocol Status

- Architecture frozen before final evaluation: **Yes**
- Hyperparameters frozen before final evaluation: **Yes**
- Three-seed development robustness confirmed: **Yes**
- Final refit used 181 writers: **Yes**
- Untouched 45-writer monitor evaluated once: **Yes**
- Monitor used for model selection: **No**
- Post-monitor tuning allowed: **No**
- Validation split used: **No**
- Official test split used: **No**

The DINOv2-S representation pivot is therefore considered **supported**, and Notebook 27 is closed without additional post-hoc tuning.